<a href="https://colab.research.google.com/github/pegumzs/SSD_Trabalho_Final/blob/main/MVP_Olist_Entregas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — Pipeline de Dados e Suporte à Decisão para os Atrasos nas Entregas do E-commerce Brasileiro (Olist)

**Universidade de Brasília (UnB) — Departamento de Engenharia de Produção**<br>

**Aluno(a):** *Pedro Augusto* · **Matrícula:** *180114425*<br>
**Plataforma:** Python 3 + Jupyter, com execução local (sem Google Drive); os dados ficam na pasta `dados/` do projeto, em camadas bronze, prata e ouro

> **Como executar:** instale as bibliotecas (`pip install -r requirements.txt`, na raiz do repositório), abra este notebook no Jupyter ou no VS Code e execute todas as células (*Run All*). Na primeira execução, o notebook baixa a base do Kaggle (sem chave de API) ou, se o Kaggle não responder, da cópia publicada pela própria Olist no GitHub; precisa de internet só nesse passo. Os dados ficam na pasta `dados/`, os gráficos e tabelas em `evidencias/` e o catálogo em `catalogo/`. O pipeline completo leva poucos minutos. A análise usa **pandas**, **numpy** e **matplotlib**; só a seção 7.3.3 usa a biblioteca **scikit-learn** (modelos de risco de atraso).

### Mapa do notebook × itens do edital

| Seção | Conteúdo | Item do edital |
|---|---|---|
| 1 | Objetivo: problema, hipótese, perguntas P1 a P5 e as três dimensões do MVP | 2.2 Objetivo · 1.2 Projetos inovadores |
| 2 | Plataforma e arquitetura do pipeline (execução local, em camadas) | 2.3 Plataforma |
| 3 | Busca dos dados, ética, licenciamento e LGPD | 3.1 Busca · 4. Ética |
| 4 | Coleta dos dados e persistência na camada bronze | 3.2 Coleta |
| 5 | Transformação e carga (ETL): camadas prata e ouro | 3.4 Carga |
| 6 | Modelagem (esquema estrela) e catálogo de dados | 3.3 Modelagem |
| 7 | Análise: 7.1 qualidade dos dados · 7.2 solução do problema · 7.3 linha de tomada de decisão, simulador e risco de atraso por pedido (machine learning) | 3.5 Análise |
| 8 | Autoavaliação e resumo para apresentação | 6. Autoavaliação |

## 1. Objetivo

### 1.1 Contexto e problema

No e-commerce, a entrega é o momento em que a promessa feita ao cliente é cumprida ou quebrada. Para a Engenharia de Produção, a entrega é um **processo** como outro qualquer: tem etapas, tempos de ciclo, gargalos e variabilidade. Cada atraso gera custo direto (atendimento, reentrega, cancelamento) e custo indireto (avaliação negativa, perda de recompra e de reputação).

A Olist é um *marketplace* que conecta pequenos vendedores aos grandes canais de venda online do Brasil. O vendedor prepara e posta o pedido; uma transportadora leva o pacote até o cliente; e, no momento da compra, o cliente recebe uma **data estimada de entrega**.

**Problema:** não se sabe *onde*, *quando* e *em qual etapa do processo* os atrasos se concentram, nem quanto eles custam em satisfação do cliente. Sem esse diagnóstico, as ações tendem a ser genéricas (por exemplo, "aumentar o prazo de todo mundo") e caras.

**Objetivo do MVP:** construir um pipeline de dados (busca → coleta → transformação e carga → modelagem → análise) que produza um diagnóstico dos atrasos de entrega e o transforme em uma **linha de tomada de decisão**: regras que indicam o que fazer em cada UF e um simulador que mostra o efeito de cada opção, como pede um sistema de suporte à decisão (SSD).

### 1.2 Hipótese do MVP e critérios de verificação

> **Hipótese (H):** os atrasos de entrega **não são aleatórios**. Eles se concentram em algumas UFs de destino e em meses de pico, crescem quando o pedido sai do estado do vendedor, nascem principalmente na etapa de transporte e reduzem a satisfação do cliente.

- **Métrica principal:** taxa de atraso = % de pedidos entregues **depois** da data prometida ao cliente.
- **Métrica de impacto:** nota média da avaliação do cliente (escala de 1 a 5).

**Critérios definidos antes da análise** (só médias, taxas e proporções):

| Critério | Pergunta | Regra |
|---|---|---|
| C1 | P1 | A UF com maior taxa de atraso tem taxa de pelo menos **2 vezes** a taxa nacional |
| C2 | P2 | Ao menos um mês tem taxa de atraso de pelo menos **2 vezes** a mediana das taxas mensais |
| C3 | P3 | O transporte responde por **mais de 50%** do tempo médio de entrega **e** a taxa de atraso dos pedidos postados fora do prazo pelo vendedor é de pelo menos **2 vezes** a dos postados no prazo |
| C4 | P4 | A taxa de atraso dos envios interestaduais é de pelo menos **1,5 vez** a dos envios locais (mesma UF) |
| C5 | P5 | A nota média dos pedidos atrasados é pelo menos **1 ponto menor** que a dos pedidos entregues no prazo |

**Regra de decisão:** a hipótese é **confirmada** se os cinco critérios forem atendidos, **parcialmente confirmada** se três ou quatro forem atendidos e **rejeitada** se dois ou menos forem atendidos.

### 1.3 Perguntas de negócio

| # | Pergunta | Decisão que a resposta apoia |
|---|---|---|
| **P1** | Quais UFs de destino têm as maiores taxas de atraso e quais concentram a maior parte dos pedidos atrasados? | Onde priorizar ações: revisão do prazo prometido, troca ou renegociação de transportadoras |
| **P2** | Como a taxa de atraso evoluiu mês a mês entre jan/2017 e ago/2018 e em quais meses houve picos? | Planejamento de capacidade logística para períodos de pico (ex.: Black Friday) |
| **P3** | Em qual etapa do processo (aprovação do pagamento, manuseio pelo vendedor ou transporte) está o gargalo do tempo de entrega, e quanto a postagem fora do prazo pelo vendedor aumenta a taxa de atraso? | Onde atuar: acordo de nível de serviço (SLA) com vendedores ou com transportadoras |
| **P4** | Qual a relação entre o escopo geográfico do envio (entrega na mesma UF vs. interestadual e entre regiões) com o tempo de entrega, o valor do frete e a taxa de atraso? | Política de prazo e de frete por rota; incentivo a vendedores e estoques próximos do cliente |
| **P5** | Quanto o atraso reduz a nota de avaliação do cliente? | Justificar o investimento em logística pelo custo da insatisfação |

### 1.4 As três dimensões do MVP

| Dimensão | Pergunta central (edital) | Como este MVP a evidencia |
|---|---|---|
| **Viabilidade técnica** | A solução pode ser construída com a tecnologia disponível? | Pipeline executado de ponta a ponta em um computador comum, com os dados gravados em camadas e relidos do disco (seções 4 e 5) |
| **Viabilidade financeira** | A solução é economicamente sustentável? | Custo de operação de R$ 0,00 (Python e bibliotecas gratuitas, sem serviço pago); o espaço ocupado e o tempo de execução são medidos no fim do notebook (seção 8.4) |
| **Desejabilidade** | A solução é o que os usuários de fato querem? | Cinco perguntas ligadas a decisões reais do gestor de logística do *marketplace*, respondidas com dados e discutidas criticamente (seção 7.2) |

## 2. Plataforma e arquitetura do pipeline

O notebook roda **localmente**, em qualquer computador com Python 3 e Jupyter (ou VS Code), sem depender do Google Drive nem de conta em serviço de nuvem. Todos os caminhos são relativos à pasta do projeto: a primeira execução baixa a base e cria a pasta `dados/`, organizada em camadas (a mesma ideia da "arquitetura medalhão" usada no Databricks). O mesmo código também roda sem alterações no Google Colab ou em um notebook do Databricks.

```
KAGGLE (fonte pública, arquivos CSV)  ou  cópia da própria Olist no GitHub
   |
   |  EXTRAÇÃO ............ kagglehub (download sem chave de API)
   v
PASTA DO PROJETO
   |-- dados/1_bronze/   5 arquivos originais, sem nenhuma alteração + registro da coleta
   |       |  TRANSFORMAÇÃO ... pandas (regras de negócio R1 a R12)
   |       v
   |-- dados/2_prata/    tabelas limpas e padronizadas, uma por conceito
   |       |  CARGA ........... esquema estrela gravado em CSV e relido do disco
   |       v
   |-- dados/3_ouro/     fato_entregas + dim_cliente + dim_vendedor + dim_tempo
   |-- catalogo/         catálogo de dados, diagrama do modelo e perfil de qualidade
   |-- evidencias/       gráficos e tabelas gerados na análise
   |
   |  ANÁLISE ............. pandas (merge, groupby, médias e taxas) + matplotlib + scikit-learn
   v
Respostas às perguntas P1 a P5 e linha de tomada de decisão
```

| Ferramenta | Papel no pipeline |
|---|---|
| Python 3 + Jupyter | Ambiente de execução local (o mesmo código roda no Colab ou no Databricks) |
| Pastas `dados/1_bronze`, `2_prata` e `3_ouro` | Armazenamento das camadas, em CSV |
| kagglehub | Extração da base pública do Kaggle (a cópia da Olist no GitHub é a segunda opção) |
| pandas / numpy | Limpeza, transformação, montagem do esquema estrela e análise |
| matplotlib | Gráficos das respostas, salvos na pasta `evidencias` |
| scikit-learn | Modelos de risco de atraso por pedido (seção 7.3.3) |

**Por que camadas?** A camada bronze guarda o dado exatamente como foi recebido, o que permite reprocessar tudo se uma regra mudar. A prata guarda o dado limpo por conceito. A ouro guarda o modelo pronto para análise. Assim a **linhagem** fica rastreável: cada número da análise pode ser seguido até o arquivo bruto de origem.

### 2.1 Configuração do ambiente

Importação das bibliotecas e definição de um padrão visual único para todos os gráficos (a mesma cor tem sempre o mesmo significado).

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Gráficos
import matplotlib.pyplot as plt

# Bibliotecas que já vêm com o Python
import os          # caminhos de pastas e arquivos
import shutil      # cópia de arquivos
import time        # cronômetro do pipeline
import urllib.request  # download da segunda fonte (GitHub)
import warnings    # controle de avisos
from datetime import datetime

# Marca o início do pipeline para medir o tempo total de execução (viabilidade técnica)
inicio_pipeline = time.time()

# Exibição das tabelas no notebook
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

# Padrão visual dos gráficos
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.grid.axis'] = 'y'      # só linhas horizontais de grade
plt.rcParams['axes.axisbelow'] = True     # grade atrás das barras
plt.rcParams['grid.color'] = '#e1e0d9'
plt.rcParams['grid.linewidth'] = 0.8

# Cores: cada cor tem sempre o mesmo significado
cor_azul = '#2a78d6'     # série principal / pedidos no prazo
cor_laranja = '#eb6834'  # destaque / pedidos atrasados
cor_cinza = '#c3c2b7'    # itens sem destaque
cor_texto = '#52514e'    # linhas de referência

print('Ambiente configurado!')
print('Versão do pandas:', pd.__version__)

### 2.2 Pastas do projeto (camadas do pipeline)

A célula abaixo localiza a pasta do projeto e cria uma subpasta para cada camada. Quando o notebook é aberto de dentro da pasta `notebooks/` do repositório, tudo é gravado na raiz do repositório: os dados em `dados/`, os gráficos e tabelas em `evidencias/` e o catálogo em `catalogo/`.

In [ ]:
# Pasta do projeto = raiz do repositório (o notebook fica na pasta notebooks/)
pasta_atual = os.getcwd()
if os.path.basename(pasta_atual) == 'notebooks':
    pasta_projeto = os.path.dirname(pasta_atual)
else:
    pasta_projeto = pasta_atual

# Camadas do pipeline + pastas de documentação
pasta_dados = os.path.join(pasta_projeto, 'dados')
pasta_bronze = os.path.join(pasta_dados, '1_bronze')           # dados brutos
pasta_prata = os.path.join(pasta_dados, '2_prata')             # dados limpos
pasta_ouro = os.path.join(pasta_dados, '3_ouro')               # esquema estrela
pasta_catalogo = os.path.join(pasta_projeto, 'catalogo')       # catálogo e diagrama
pasta_evidencias = os.path.join(pasta_projeto, 'evidencias')   # gráficos e tabelas

# Cria as pastas (se já existirem, não dá erro)
lista_pastas = [pasta_bronze, pasta_prata, pasta_ouro, pasta_catalogo, pasta_evidencias]
for pasta in lista_pastas:
    os.makedirs(pasta, exist_ok=True)

print('Pasta do projeto:', pasta_projeto)
for pasta in lista_pastas:
    print('   ', os.path.relpath(pasta, pasta_projeto))

## 3. Busca dos dados, ética, licenciamento e LGPD

### 3.1 Base escolhida

| Item | Descrição |
|---|---|
| **Base** | *Brazilian E-Commerce Public Dataset by Olist* |
| **Link** | https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce |
| **Identificador no kagglehub** | `olistbr/brazilian-ecommerce` |
| **Publicador** | Olist (dados comerciais reais, anonimizados pela própria empresa) |
| **Período** | Compras de set/2016 a out/2018 — cerca de 100 mil pedidos |
| **Formato original** | 9 arquivos CSV (UTF-8, separador vírgula), em modelo relacional |
| **Licença** | **CC BY-NC-SA 4.0** (Creative Commons Atribuição–NãoComercial–CompartilhaIgual 4.0) |

**Por que esta base atende ao objetivo (e não o contrário):** as perguntas da seção 1 exigem (a) todas as datas do processo de entrega — compra, aprovação, postagem, entrega e data prometida —, (b) o prazo limite de postagem do vendedor, (c) a UF de clientes e vendedores, (d) o valor do frete e (e) a nota de avaliação do cliente. Esta base tem exatamente essas variáveis, com dados reais e em volume suficiente. Ela também é relacional, o que exige **junção e conciliação de conjuntos de dados diferentes** (pedidos × itens × avaliações).

**Tabelas usadas (5 de 9):**

| Arquivo | Conteúdo | Uso no MVP |
|---|---|---|
| `olist_orders_dataset.csv` | 1 linha por pedido, com status e as 5 datas do processo | Atraso, etapas do processo, período |
| `olist_order_items_dataset.csv` | 1 linha por item do pedido | Vendedor, prazo limite de postagem, preço e frete |
| `olist_customers_dataset.csv` | Localização do cliente (CEP, cidade, UF) | UF e região de destino |
| `olist_sellers_dataset.csv` | Localização do vendedor (CEP, cidade, UF) | UF e região de origem |
| `olist_order_reviews_dataset.csv` | Avaliação do cliente (nota de 1 a 5) | Impacto do atraso na satisfação |

**Tabelas não usadas:** `olist_geolocation_dataset.csv` (1 milhão de coordenadas por CEP), `olist_order_payments_dataset.csv`, `olist_products_dataset.csv` e `product_category_name_translation.csv`. Pelo princípio do MVP ("nem mais, nem menos"), elas não são necessárias: a P4 compara o **escopo geográfico** do envio (mesma UF × interestadual × outra região), e a UF já está nas tabelas de clientes e vendedores.

### 3.2 Licenciamento

A licença **CC BY-NC-SA 4.0** permite copiar, redistribuir e criar obras derivadas **para fins não comerciais**, desde que se dê crédito à Olist e que as obras derivadas mantenham a mesma licença. O uso acadêmico deste trabalho é permitido. Mesmo assim, os dados brutos **não** são versionados no repositório do GitHub (a pasta `dados/` está no `.gitignore`): o notebook baixa a base na primeira execução, o que garante a reprodutibilidade sem redistribuir os arquivos.

### 3.3 Ética e proteção de dados (LGPD — Lei nº 13.709/2018)

- **Dados pessoais:** os identificadores de pedido, cliente e vendedor são códigos *hash*. Não há nome, CPF, e-mail nem endereço completo: só o prefixo de 5 dígitos do CEP, a cidade e a UF.
- **Minimização (LGPD, art. 6º, III — princípio da necessidade):** os títulos e comentários em texto livre das avaliações e o identificador único do cliente (`customer_unique_id`) são descartados na camada prata, porque não são necessários às perguntas e poderiam permitir a identificação de pessoas.
- **Resultados agregados:** todas as respostas são apresentadas por UF, mês, etapa ou faixa, nunca por cliente.

## 4. Coleta dos dados (extração → camada bronze)

A coleta tenta duas fontes, nesta ordem:

1. **Kaggle**, a fonte oficial da base, pela biblioteca `kagglehub`, que baixa bases **públicas** sem exigir conta ou chave de API;
2. **GitHub**, se o Kaggle não responder ou se o `kagglehub` não estiver instalado: a cópia dos mesmos arquivos publicada pela própria Olist no repositório [`olist/work-at-olist-data`](https://github.com/olist/work-at-olist-data). Os arquivos são iguais aos do Kaggle (mesmo número de linhas e de colunas, conferido no registro da coleta).

A data e a hora da coleta e a fonte usada são registradas automaticamente, junto com o volume e o formato de cada arquivo, como pede o item 3.2 do edital.

> **Sem internet?** Baixe o ZIP pelo botão *Download* da página da base no Kaggle, descompacte os arquivos na pasta `dados/0_download/` e execute o notebook normalmente: se o Kaggle não responder, a célula abaixo usa os arquivos que já estiverem nessa pasta.

### 4.1 Extração

In [ ]:
# Os 5 arquivos que respondem às perguntas do MVP
arquivos_usados = [
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_order_reviews_dataset.csv'
]

# Pasta para downloads que não vêm do kagglehub (GitHub ou ZIP baixado à mão)
pasta_download = os.path.join(pasta_dados, '0_download')
url_github = 'https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/'

try:
    # 1ª opção: Kaggle, via kagglehub (fonte oficial)
    warnings.filterwarnings('ignore', message='IProgress not found')   # aviso só visual da barra de progresso
    import kagglehub
    caminho_download = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    fonte_coleta = 'Kaggle via kagglehub'
    origem_coleta = 'kaggle.com/datasets/olistbr/brazilian-ecommerce'
except Exception as erro:
    print('Kaggle indisponível (' + type(erro).__name__ + '). Usando a segunda fonte.')
    os.makedirs(pasta_download, exist_ok=True)
    caminho_download = pasta_download
    arquivos_faltando = [nome for nome in arquivos_usados
                         if not os.path.exists(os.path.join(pasta_download, nome))]
    if len(arquivos_faltando) == 0:
        # Arquivos já estão em dados/0_download (ZIP do Kaggle baixado à mão ou execução anterior)
        fonte_coleta = 'arquivos locais em dados/0_download'
        origem_coleta = 'kaggle.com/datasets/olistbr/brazilian-ecommerce (cópia local)'
    else:
        # 2ª opção: cópia publicada pela própria Olist no GitHub
        for nome_arquivo in arquivos_faltando:
            print('   baixando', nome_arquivo, '...')
            urllib.request.urlretrieve(url_github + nome_arquivo, os.path.join(pasta_download, nome_arquivo))
        fonte_coleta = 'GitHub da Olist'
        origem_coleta = 'github.com/olist/work-at-olist-data (cópia da Olist)'

# Registra a data e a hora da coleta (relógio do computador)
data_coleta = datetime.now()
texto_data_coleta = data_coleta.strftime('%d/%m/%Y')

print('Fonte usada:', fonte_coleta)
print('Base baixada em:', caminho_download)
print('Data/hora da coleta:', data_coleta.strftime('%d/%m/%Y %H:%M'))
print()
print('Arquivos disponíveis na fonte:')
for nome_arquivo in sorted(os.listdir(caminho_download)):
    print('   ', nome_arquivo)

### 4.2 Persistência na camada bronze e registro da coleta

Regra da camada bronze: os 5 arquivos usados são copiados para a pasta `dados/1_bronze` **exatamente como vieram**, sem nenhuma alteração. Para cada arquivo, o registro da coleta guarda o volume (linhas, colunas e tamanho), o formato original, a origem e a data da coleta.

In [ ]:
# A lista arquivos_usados foi definida na seção 4.1
lista_registro = []

for nome_arquivo in arquivos_usados:
    # Copia o arquivo bruto da pasta do download para a camada bronze (dados/1_bronze)
    caminho_origem = os.path.join(caminho_download, nome_arquivo)
    caminho_destino = os.path.join(pasta_bronze, nome_arquivo)
    shutil.copy(caminho_origem, caminho_destino)

    # Lê a cópia gravada na camada bronze só para medir o volume
    tabela_temporaria = pd.read_csv(caminho_destino)
    tamanho_mb = os.path.getsize(caminho_destino) / (1024 * 1024)

    lista_registro.append({
        'arquivo': nome_arquivo,
        'formato_original': 'CSV (UTF-8, separador vírgula)',
        'linhas': len(tabela_temporaria),
        'colunas': tabela_temporaria.shape[1],
        'tamanho_mb': round(tamanho_mb, 2),
        'data_hora_coleta': data_coleta.strftime('%Y-%m-%d %H:%M'),
        'origem': origem_coleta
    })

df_registro_coleta = pd.DataFrame(lista_registro)

# O registro da coleta também fica salvo na camada bronze (reprodutibilidade)
df_registro_coleta.to_csv(os.path.join(pasta_bronze, 'registro_coleta.csv'), index=False)

print('Arquivos copiados para a camada bronze:', len(df_registro_coleta))
print('Total de linhas coletadas:', df_registro_coleta['linhas'].sum())
print('Volume total (MB):', round(df_registro_coleta['tamanho_mb'].sum(), 1))
df_registro_coleta

In [ ]:
# Evidência da persistência: listagem da camada bronze, lida do disco
print('Conteúdo de', pasta_bronze)
for nome_arquivo in sorted(os.listdir(pasta_bronze)):
    tamanho_mb = os.path.getsize(os.path.join(pasta_bronze, nome_arquivo)) / (1024 * 1024)
    print(f'   {nome_arquivo:40s} {tamanho_mb:6.2f} MB')

## 5. Transformação e carga (ETL)

O ETL segue o caminho **bronze → prata → ouro**:

1. **Extração:** leitura dos 5 arquivos brutos da camada bronze, na pasta `dados/1_bronze` (seção 5.1).
2. **Transformação:** limpeza e padronização com pandas, guardadas na camada prata (seções 5.2 e 5.3), e construção das tabelas do esquema estrela (seção 5.4).
3. **Carga:** gravação da fato e das dimensões em CSV na camada ouro (`dados/3_ouro`) e releitura dos arquivos para comprovar a persistência (seção 5.5).

### 5.1 Leitura da camada bronze

A partir daqui o pipeline lê os dados **da camada bronze**, e não da pasta do download. As colunas de data já são lidas como datas (`parse_dates`), o que não altera o arquivo bruto.

In [ ]:
colunas_data_pedidos = [
    'order_purchase_timestamp',        # compra
    'order_approved_at',               # aprovação do pagamento
    'order_delivered_carrier_date',    # postagem na transportadora
    'order_delivered_customer_date',   # entrega ao cliente
    'order_estimated_delivery_date'    # data prometida ao cliente
]

df_pedidos = pd.read_csv(os.path.join(pasta_bronze, 'olist_orders_dataset.csv'),
                         parse_dates=colunas_data_pedidos)

df_itens = pd.read_csv(os.path.join(pasta_bronze, 'olist_order_items_dataset.csv'),
                       parse_dates=['shipping_limit_date'])

df_clientes = pd.read_csv(os.path.join(pasta_bronze, 'olist_customers_dataset.csv'))

df_vendedores = pd.read_csv(os.path.join(pasta_bronze, 'olist_sellers_dataset.csv'))

df_avaliacoes = pd.read_csv(os.path.join(pasta_bronze, 'olist_order_reviews_dataset.csv'),
                            parse_dates=['review_creation_date', 'review_answer_timestamp'])

print('pedidos    :', df_pedidos.shape)
print('itens      :', df_itens.shape)
print('clientes   :', df_clientes.shape)
print('vendedores :', df_vendedores.shape)
print('avaliacoes :', df_avaliacoes.shape)
df_pedidos.head(3)

### 5.2 Regras de negócio aplicadas na transformação

As regras estão numeradas para que possam ser citadas no catálogo de dados (linhagem), na análise de qualidade (seção 7.1) e na discussão dos resultados. Elas afetam diretamente a interpretação dos números.

| Regra | Tipo | Descrição | Camada |
|---|---|---|---|
| **R1** | Padronização de nomes | Colunas renomeadas para português, em minúsculas e sem espaços | prata |
| **R2** | Padronização de chaves | Prefixo de CEP convertido para **texto com 5 dígitos** (ex.: 1310 → "01310"): o CSV gravou o CEP como número e perdeu o zero à esquerda | prata |
| **R3** | Enriquecimento | Região geográfica (IBGE) derivada da UF | prata |
| **R4** | Minimização (LGPD) | Descarte do título e do comentário das avaliações e do `customer_unique_id` | prata |
| **R5** | Deduplicação | Pedido com mais de uma avaliação: fica a de **resposta mais recente** | prata |
| **R6** | Filtro de escopo | Entram na fato só pedidos com status `delivered` **e** data de entrega preenchida | ouro |
| **R7** | Conciliação do grão (pedidos × itens) | Os itens são agregados por pedido (quantidade, soma do preço, soma do frete e maior prazo limite de postagem). Pedidos com itens de **mais de um vendedor** são excluídos: a data de entrega é única por pedido, então o atraso não poderia ser atribuído a um só vendedor | ouro |
| **R8** | Conversão de unidades | Diferenças entre datas convertidas de segundos para **dias** (÷ 86.400), com fração de dia | ouro |
| **R9** | Tratamento de inconsistência | Etapa com duração negativa (datas fora de ordem) vira nula; o pedido é mantido | ouro |
| **R10** | Regra de negócio | **Atraso** = a *data* (dia) da entrega é posterior à data estimada. Entrega no próprio dia estimado conta como "no prazo", porque a data estimada não tem horário (vem sempre com 00:00:00) | ouro |
| **R11** | Regra de negócio | **Postagem atrasada** = o vendedor entregou o pacote à transportadora depois do prazo limite de postagem (`shipping_limit_date`) | ouro |
| **R12** | Regra de negócio | **Escopo geográfico do envio:** `tipo_envio` = *Local* (UF do vendedor = UF do cliente) ou *Interestadual*; `escopo_envio` = *Mesma UF*, *Outra UF da mesma região* ou *Outra região* | ouro |

### 5.3 Limpeza e padronização (camada prata) — regras R1 a R5

In [ ]:
# ------------------------------------------------------------
# 5.3.1 Clientes e vendedores — regras R1, R2, R3 e R4
# ------------------------------------------------------------
# Dicionário UF -> região do IBGE (regra R3)
regiao_por_uf = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste',
    'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

# --- Clientes ---
clientes_prata = df_clientes.rename(columns={
    'customer_id': 'id_cliente',
    'customer_zip_code_prefix': 'cep_prefixo',
    'customer_city': 'cidade',
    'customer_state': 'uf'
})
# R4: o identificador único da pessoa não é necessário às perguntas (minimização - LGPD)
clientes_prata = clientes_prata.drop(columns=['customer_unique_id'])
# R2: CEP como texto de 5 dígitos (devolve o zero à esquerda)
clientes_prata['cep_prefixo'] = clientes_prata['cep_prefixo'].astype(str).str.zfill(5)
# R3: região a partir da UF
clientes_prata['regiao'] = clientes_prata['uf'].map(regiao_por_uf)

# --- Vendedores ---
vendedores_prata = df_vendedores.rename(columns={
    'seller_id': 'id_vendedor',
    'seller_zip_code_prefix': 'cep_prefixo',
    'seller_city': 'cidade',
    'seller_state': 'uf'
})
vendedores_prata['cep_prefixo'] = vendedores_prata['cep_prefixo'].astype(str).str.zfill(5)
vendedores_prata['regiao'] = vendedores_prata['uf'].map(regiao_por_uf)

# Exemplo da regra R2: o primeiro cliente cujo CEP perdeu o zero à esquerda
cep_sem_zero = df_clientes['customer_zip_code_prefix'] < 10000
posicao_exemplo = cep_sem_zero.idxmax()   # posição do primeiro valor True
print('Exemplo da regra R2 (padronização do CEP):')
print('   antes :', df_clientes.loc[posicao_exemplo, 'customer_zip_code_prefix'])
print('   depois:', clientes_prata.loc[posicao_exemplo, 'cep_prefixo'])
print('CEPs de clientes que recuperaram o zero à esquerda  :', cep_sem_zero.sum())
print('CEPs de vendedores que recuperaram o zero à esquerda:', (df_vendedores['seller_zip_code_prefix'] < 10000).sum())
print('Clientes sem região (UF inválida):', clientes_prata['regiao'].isnull().sum())
clientes_prata.head(3)

In [ ]:
# ------------------------------------------------------------
# 5.3.2 Pedidos e itens — regra R1 (nomes em português)
# ------------------------------------------------------------
pedidos_prata = df_pedidos.rename(columns={
    'order_id': 'id_pedido',
    'customer_id': 'id_cliente',
    'order_status': 'status',
    'order_purchase_timestamp': 'data_compra',
    'order_approved_at': 'data_aprovacao',
    'order_delivered_carrier_date': 'data_postagem',
    'order_delivered_customer_date': 'data_entrega',
    'order_estimated_delivery_date': 'data_estimada'
})

itens_prata = df_itens.rename(columns={
    'order_id': 'id_pedido',
    'order_item_id': 'numero_item',
    'product_id': 'id_produto',
    'seller_id': 'id_vendedor',
    'shipping_limit_date': 'prazo_limite_postagem',
    'price': 'preco',
    'freight_value': 'valor_frete'
})

print('Colunas de pedidos:', list(pedidos_prata.columns))
print('Colunas de itens  :', list(itens_prata.columns))

In [ ]:
# ------------------------------------------------------------
# 5.3.3 Avaliações — minimização (R4), nomes (R1) e deduplicação (R5)
# ------------------------------------------------------------
# R4: ficam só as colunas necessárias; título e comentário em texto livre são descartados
avaliacoes_prata = df_avaliacoes[['review_id', 'order_id', 'review_score',
                                  'review_creation_date', 'review_answer_timestamp']]

avaliacoes_prata = avaliacoes_prata.rename(columns={
    'review_id': 'id_avaliacao',
    'order_id': 'id_pedido',
    'review_score': 'nota',
    'review_creation_date': 'data_criacao_avaliacao',
    'review_answer_timestamp': 'data_resposta_avaliacao'
})

# R5: se o pedido tem mais de uma avaliação, fica a de resposta mais recente
qtd_antes = len(avaliacoes_prata)
avaliacoes_prata = avaliacoes_prata.sort_values('data_resposta_avaliacao')
avaliacoes_prata = avaliacoes_prata.drop_duplicates(subset='id_pedido', keep='last')

print('Avaliações antes da deduplicação:', qtd_antes)
print('Avaliações depois (1 por pedido):', len(avaliacoes_prata))
print('Linhas removidas pela regra R5  :', qtd_antes - len(avaliacoes_prata))

In [ ]:
# ------------------------------------------------------------
# 5.3.4 Persistência da camada PRATA (dados/2_prata)
# ------------------------------------------------------------
pedidos_prata.to_csv(os.path.join(pasta_prata, 'pedidos.csv'), index=False)
itens_prata.to_csv(os.path.join(pasta_prata, 'itens_pedido.csv'), index=False)
clientes_prata.to_csv(os.path.join(pasta_prata, 'clientes.csv'), index=False)
vendedores_prata.to_csv(os.path.join(pasta_prata, 'vendedores.csv'), index=False)
avaliacoes_prata.to_csv(os.path.join(pasta_prata, 'avaliacoes.csv'), index=False)

print('Camada prata salva em:', pasta_prata)
for nome_arquivo in sorted(os.listdir(pasta_prata)):
    tamanho_mb = os.path.getsize(os.path.join(pasta_prata, nome_arquivo)) / (1024 * 1024)
    print(f'   {nome_arquivo:25s} {tamanho_mb:6.2f} MB')

### 5.4 Construção das tabelas do esquema estrela

O modelo é explicado na seção 6. Aqui ficam os passos de construção: primeiro as três dimensões, depois a tabela fato, montada em três passos (escopo e grão → medidas de tempo → avaliação, escopo geográfico e chave de tempo).

In [ ]:
# ------------------------------------------------------------
# 5.4.1 Dimensão TEMPO (calendário diário de todo o período da base)
# ------------------------------------------------------------
primeiro_dia = pedidos_prata['data_compra'].min().normalize()
ultimo_dia = pedidos_prata['data_compra'].max().normalize()
datas_calendario = pd.date_range(start=primeiro_dia, end=ultimo_dia, freq='D')

dim_tempo = pd.DataFrame({'data': datas_calendario})
dim_tempo['id_tempo'] = dim_tempo['data'].dt.strftime('%Y%m%d').astype(int)   # chave AAAAMMDD
dim_tempo['ano'] = dim_tempo['data'].dt.year
dim_tempo['mes'] = dim_tempo['data'].dt.month
dim_tempo['ano_mes'] = dim_tempo['data'].dt.strftime('%Y-%m')
dim_tempo['trimestre'] = dim_tempo['data'].dt.quarter

nomes_dias = {0: 'segunda', 1: 'terça', 2: 'quarta', 3: 'quinta',
              4: 'sexta', 5: 'sábado', 6: 'domingo'}
dim_tempo['dia_semana'] = dim_tempo['data'].dt.dayofweek.map(nomes_dias)

# A data vira texto no formato AAAA-MM-DD para ser gravada no CSV
dim_tempo['data'] = dim_tempo['data'].dt.strftime('%Y-%m-%d')

# Ordem das colunas: chave primeiro
dim_tempo = dim_tempo[['id_tempo', 'data', 'ano', 'mes', 'ano_mes', 'trimestre', 'dia_semana']]

print('dim_tempo:', dim_tempo.shape)
dim_tempo.head(3)

In [ ]:
# ------------------------------------------------------------
# 5.4.2 Dimensões CLIENTE e VENDEDOR (vêm direto da camada prata)
# ------------------------------------------------------------
dim_cliente = clientes_prata[['id_cliente', 'cep_prefixo', 'cidade', 'uf', 'regiao']].copy()
dim_vendedor = vendedores_prata[['id_vendedor', 'cep_prefixo', 'cidade', 'uf', 'regiao']].copy()

print('dim_cliente :', dim_cliente.shape)
print('dim_vendedor:', dim_vendedor.shape)
dim_vendedor.head(3)

In [ ]:
# ------------------------------------------------------------
# 5.4.3 Tabela FATO — passo 1: escopo (R6) e conciliação do grão pedidos x itens (R7)
# ------------------------------------------------------------
# R6: só pedidos com status "delivered" e com data de entrega preenchida
filtro_entregues = (pedidos_prata['status'] == 'delivered') & (pedidos_prata['data_entrega'].notnull())
fato = pedidos_prata[filtro_entregues].copy()
print('Pedidos entregues com data de entrega:', len(fato))

# R7: a tabela de itens tem 1 linha por item; a fato tem 1 linha por pedido.
# Por isso os itens são resumidos por pedido antes da junção.
itens_por_pedido = itens_prata.groupby('id_pedido').agg(
    qtd_itens=('numero_item', 'count'),
    qtd_vendedores=('id_vendedor', 'nunique'),
    id_vendedor=('id_vendedor', 'first'),
    valor_produtos=('preco', 'sum'),
    valor_frete=('valor_frete', 'sum'),
    prazo_limite_postagem=('prazo_limite_postagem', 'max')
)
itens_por_pedido = itens_por_pedido.reset_index()

fato = fato.merge(itens_por_pedido, on='id_pedido', how='left')

# R7: o grão da fato é "1 pedido entregue de 1 único vendedor"
qtd_multivendedor = (fato['qtd_vendedores'] > 1).sum()
fato = fato[fato['qtd_vendedores'] == 1].copy()

print('Pedidos com mais de um vendedor excluídos (R7):', qtd_multivendedor)
print('Pedidos que seguem para a tabela fato         :', len(fato))

In [ ]:
# ------------------------------------------------------------
# 5.4.4 Tabela FATO — passo 2: medidas de tempo (regras R8, R9, R10 e R11)
# ------------------------------------------------------------
segundos_por_dia = 24 * 60 * 60   # R8: conversão de segundos para dias

# Etapas do processo de entrega (em dias, com fração de dia)
fato['dias_aprovacao'] = (fato['data_aprovacao'] - fato['data_compra']).dt.total_seconds() / segundos_por_dia
fato['dias_manuseio'] = (fato['data_postagem'] - fato['data_aprovacao']).dt.total_seconds() / segundos_por_dia
fato['dias_transporte'] = (fato['data_entrega'] - fato['data_postagem']).dt.total_seconds() / segundos_por_dia
fato['dias_entrega_total'] = (fato['data_entrega'] - fato['data_compra']).dt.total_seconds() / segundos_por_dia
fato['dias_prometidos'] = (fato['data_estimada'] - fato['data_compra']).dt.total_seconds() / segundos_por_dia

# R9: etapa com duração negativa = datas fora de ordem -> a etapa fica nula
print('Etapas de aprovação negativas anuladas :', (fato['dias_aprovacao'] < 0).sum())
print('Etapas de manuseio negativas anuladas  :', (fato['dias_manuseio'] < 0).sum())
print('Etapas de transporte negativas anuladas:', (fato['dias_transporte'] < 0).sum())
fato.loc[fato['dias_aprovacao'] < 0, 'dias_aprovacao'] = np.nan
fato.loc[fato['dias_manuseio'] < 0, 'dias_manuseio'] = np.nan
fato.loc[fato['dias_transporte'] < 0, 'dias_transporte'] = np.nan

# R10: atraso medido em dias corridos (a data estimada não tem horário)
data_entrega_dia = fato['data_entrega'].dt.normalize()     # zera o horário
data_estimada_dia = fato['data_estimada'].dt.normalize()
fato['dias_atraso'] = (data_entrega_dia - data_estimada_dia).dt.days
fato['flag_atraso'] = np.where(fato['dias_atraso'] > 0, 1, 0)

# R11: o vendedor postou depois do prazo limite de postagem?
fato['flag_postagem_atrasada'] = np.where(fato['data_postagem'] > fato['prazo_limite_postagem'], 1.0, 0.0)
fato.loc[fato['data_postagem'].isnull(), 'flag_postagem_atrasada'] = np.nan

# Por que a regra R10 importa: comparando data E hora, quem recebeu no dia
# prometido às 15h seria contado como atrasado
taxa_regra_r10 = 100 * fato['flag_atraso'].mean()
taxa_data_e_hora = 100 * (fato['data_entrega'] > fato['data_estimada']).mean()
print()
print(f'Taxa de atraso com a regra R10 (comparando o dia): {taxa_regra_r10:.2f}%')
print(f'Taxa se comparássemos data e hora (superestimada): {taxa_data_e_hora:.2f}%')

In [ ]:
# ------------------------------------------------------------
# 5.4.5 Tabela FATO — passo 3: avaliação, escopo geográfico (R12) e chave de tempo
# ------------------------------------------------------------
# Nota da avaliação (já deduplicada pela regra R5)
fato = fato.merge(avaliacoes_prata[['id_pedido', 'nota']], on='id_pedido', how='left')
fato = fato.rename(columns={'nota': 'nota_avaliacao'})

# R12: traz a UF e a região do cliente e do vendedor só para comparar origem e destino
local_cliente = dim_cliente[['id_cliente', 'uf', 'regiao']]
local_cliente = local_cliente.rename(columns={'uf': 'uf_cliente', 'regiao': 'regiao_cliente'})
local_vendedor = dim_vendedor[['id_vendedor', 'uf', 'regiao']]
local_vendedor = local_vendedor.rename(columns={'uf': 'uf_vendedor', 'regiao': 'regiao_vendedor'})

fato = fato.merge(local_cliente, on='id_cliente', how='left')
fato = fato.merge(local_vendedor, on='id_vendedor', how='left')

# Coluna simples pedida na P4: o vendedor está na mesma UF do cliente?
fato['tipo_envio'] = np.where(fato['uf_vendedor'] == fato['uf_cliente'], 'Local', 'Interestadual')

# Versão com três níveis (começa como "Outra região" e vai sendo corrigida)
fato['escopo_envio'] = 'Outra região'
fato.loc[fato['regiao_vendedor'] == fato['regiao_cliente'], 'escopo_envio'] = 'Outra UF da mesma região'
fato.loc[fato['uf_vendedor'] == fato['uf_cliente'], 'escopo_envio'] = 'Mesma UF'

# Chave da dimensão tempo (data da compra no formato AAAAMMDD)
fato['id_tempo_compra'] = fato['data_compra'].dt.strftime('%Y%m%d').astype(int)

# Colunas finais da fato: chaves primeiro, depois as medidas
# (UF e região NÃO ficam na fato: elas já estão nas dimensões)
colunas_fato = ['id_pedido', 'id_cliente', 'id_vendedor', 'id_tempo_compra',
                'qtd_itens', 'valor_produtos', 'valor_frete',
                'dias_aprovacao', 'dias_manuseio', 'dias_transporte', 'dias_entrega_total',
                'dias_prometidos', 'dias_atraso', 'flag_atraso', 'flag_postagem_atrasada',
                'nota_avaliacao', 'tipo_envio', 'escopo_envio']
fato_entregas = fato[colunas_fato].copy()

# Arredondamentos: valores em R$ e dias com 2 casas decimais
fato_entregas['valor_produtos'] = fato_entregas['valor_produtos'].round(2)
fato_entregas['valor_frete'] = fato_entregas['valor_frete'].round(2)
colunas_em_dias = ['dias_aprovacao', 'dias_manuseio', 'dias_transporte', 'dias_entrega_total', 'dias_prometidos']
for coluna in colunas_em_dias:
    fato_entregas[coluna] = fato_entregas[coluna].round(2)

print('fato_entregas:', fato_entregas.shape)
print('id_pedido é único?', fato_entregas['id_pedido'].is_unique)
print('Pedidos sem avaliação:', fato_entregas['nota_avaliacao'].isnull().sum())
print()
print('Pedidos por tipo de envio (R12):')
print(fato_entregas['tipo_envio'].value_counts())
fato_entregas.head(3)

### 5.5 Carga na camada ouro e verificação da persistência

As quatro tabelas do esquema estrela são gravadas em CSV na pasta `dados/3_ouro`. Em seguida, os arquivos são **lidos de volta do disco** com `pd.read_csv()`. Toda a análise da seção 7.2 usa essas tabelas relidas, e não as que estavam na memória, o que comprova que o resultado depende apenas do que ficou gravado na camada ouro.

In [ ]:
# ------------------------------------------------------------
# 5.5.1 Carga: gravação das 4 tabelas na camada OURO
# ------------------------------------------------------------
fato_entregas.to_csv(os.path.join(pasta_ouro, 'fato_entregas.csv'), index=False)
dim_cliente.to_csv(os.path.join(pasta_ouro, 'dim_cliente.csv'), index=False)
dim_vendedor.to_csv(os.path.join(pasta_ouro, 'dim_vendedor.csv'), index=False)
dim_tempo.to_csv(os.path.join(pasta_ouro, 'dim_tempo.csv'), index=False)

# ------------------------------------------------------------
# 5.5.2 Releitura das tabelas gravadas na camada ouro
# ------------------------------------------------------------
# O CSV não guarda o tipo das colunas. O CEP é lido como texto (dtype=str);
# senão o zero à esquerda se perderia de novo (regra R2).
fato_ouro = pd.read_csv(os.path.join(pasta_ouro, 'fato_entregas.csv'))
dim_cliente_ouro = pd.read_csv(os.path.join(pasta_ouro, 'dim_cliente.csv'), dtype={'cep_prefixo': str})
dim_vendedor_ouro = pd.read_csv(os.path.join(pasta_ouro, 'dim_vendedor.csv'), dtype={'cep_prefixo': str})
dim_tempo_ouro = pd.read_csv(os.path.join(pasta_ouro, 'dim_tempo.csv'))

# ------------------------------------------------------------
# 5.5.3 Verificação: o que foi gravado é igual ao que foi lido?
# ------------------------------------------------------------
verificacao_carga = pd.DataFrame({
    'tabela': ['fato_entregas', 'dim_cliente', 'dim_vendedor', 'dim_tempo'],
    'linhas_gravadas': [len(fato_entregas), len(dim_cliente), len(dim_vendedor), len(dim_tempo)],
    'linhas_lidas': [len(fato_ouro), len(dim_cliente_ouro), len(dim_vendedor_ouro), len(dim_tempo_ouro)],
    'colunas_lidas': [fato_ouro.shape[1], dim_cliente_ouro.shape[1],
                      dim_vendedor_ouro.shape[1], dim_tempo_ouro.shape[1]]
})
verificacao_carga['carga_ok'] = np.where(
    verificacao_carga['linhas_gravadas'] == verificacao_carga['linhas_lidas'], 'SIM', 'NÃO')

# Somas de controle: os totais precisam ser os mesmos antes e depois da carga
print('Soma de controle - frete total (R$):  gravado =', round(fato_entregas['valor_frete'].sum(), 2),
      '| lido =', round(fato_ouro['valor_frete'].sum(), 2))
print('Soma de controle - pedidos atrasados: gravado =', fato_entregas['flag_atraso'].sum(),
      '| lido =', fato_ouro['flag_atraso'].sum())
verificacao_carga

In [ ]:
# ------------------------------------------------------------
# 5.5.4 Integridade do esquema estrela e evidência dos arquivos gravados
# ------------------------------------------------------------
print('Chaves primárias sem repetição:')
print('   fato_entregas.id_pedido :', fato_ouro['id_pedido'].is_unique)
print('   dim_cliente.id_cliente  :', dim_cliente_ouro['id_cliente'].is_unique)
print('   dim_vendedor.id_vendedor:', dim_vendedor_ouro['id_vendedor'].is_unique)
print('   dim_tempo.id_tempo      :', dim_tempo_ouro['id_tempo'].is_unique)
print()
print('Toda chave estrangeira da fato existe na dimensão?')
print('   id_cliente      -> dim_cliente :', fato_ouro['id_cliente'].isin(dim_cliente_ouro['id_cliente']).all())
print('   id_vendedor     -> dim_vendedor:', fato_ouro['id_vendedor'].isin(dim_vendedor_ouro['id_vendedor']).all())
print('   id_tempo_compra -> dim_tempo   :', fato_ouro['id_tempo_compra'].isin(dim_tempo_ouro['id_tempo']).all())
print()
print('Arquivos gravados na camada ouro (dados/3_ouro):')
for nome_arquivo in sorted(os.listdir(pasta_ouro)):
    tamanho_mb = os.path.getsize(os.path.join(pasta_ouro, nome_arquivo)) / (1024 * 1024)
    print(f'   {nome_arquivo:22s} {tamanho_mb:6.2f} MB')

## 6. Modelagem e catálogo de dados

### 6.1 Esquema estrela

O problema é analítico (medir atrasos por local, por período, por etapa e por rota), então o modelo escolhido é um **esquema estrela**, como em um Data Warehouse: uma tabela fato central com as medidas e tabelas de dimensão com os atributos usados para filtrar e agrupar.

**Grão da tabela fato (decisão mais importante do modelo):** **1 linha = 1 pedido entregue, atendido por um único vendedor.** A data de entrega existe por pedido (não por item), por isso o pedido é o grão natural para medir atraso. Os itens foram resumidos por pedido (regra R7).

```
                          +-----------------------------+
                          |          dim_tempo          |
                          |-----------------------------|
                          | PK id_tempo  (AAAAMMDD)     |
                          |    data, ano, mes, ano_mes, |
                          |    trimestre, dia_semana    |
                          +--------------+--------------+
                                         | 1
                                         |
                                         | N
+--------------------+       +-----------+-------------------+       +--------------------+
|    dim_cliente     | 1   N |         fato_entregas         | N   1 |    dim_vendedor    |
|--------------------|-------|-------------------------------|-------|--------------------|
| PK id_cliente      |       | PK id_pedido                  |       | PK id_vendedor     |
|    cep_prefixo     |       | FK id_cliente                 |       |    cep_prefixo     |
|    cidade          |       | FK id_vendedor                |       |    cidade          |
|    uf              |       | FK id_tempo_compra            |       |    uf              |
|    regiao          |       |-------------------------------|       |    regiao          |
+--------------------+       | qtd_itens, valor_produtos,    |       +--------------------+
                             | valor_frete,                  |
                             | dias_aprovacao, dias_manuseio,|
                             | dias_transporte,              |
                             | dias_entrega_total,           |
                             | dias_prometidos, dias_atraso, |
                             | flag_atraso,                  |
                             | flag_postagem_atrasada,       |
                             | nota_avaliacao,               |
                             | tipo_envio, escopo_envio      |
                             +-------------------------------+
```

| Tabela | Tipo | Conteúdo | Chave primária |
|---|---|---|---|
| `fato_entregas` | Fato | Medidas do pedido: valores, tempos de cada etapa, atraso, nota e escopo do envio | `id_pedido` |
| `dim_cliente` | Dimensão | Localização do cliente: CEP, cidade, UF e região | `id_cliente` |
| `dim_vendedor` | Dimensão | Localização do vendedor: CEP, cidade, UF e região | `id_vendedor` |
| `dim_tempo` | Dimensão | Calendário diário: ano, mês, trimestre e dia da semana | `id_tempo` (AAAAMMDD) |

Todas as relações são **1 : N** (uma linha da dimensão para muitas linhas da fato). As dimensões são desnormalizadas (UF e região ficam dentro de cliente e vendedor), o que caracteriza um esquema estrela e não um *snowflake*. `tipo_envio` e `escopo_envio` ficam na própria fato porque descrevem a combinação de origem e destino de cada pedido, e não o cliente ou o vendedor isoladamente.

### 6.2 Catálogo de dados

O catálogo segue o conteúdo mínimo do item 3.3 do edital, com exatamente seis colunas: **Nome do atributo**, **Tipo**, **Descrição**, **Domínio**, **Obrigatoriedade** e **Linhagem**. Para identificar a tabela, o nome do atributo vem no formato `tabela.atributo`. A linhagem informa o arquivo de origem, a data de extração (registrada na coleta, seção 4) e a transformação aplicada (regras R1 a R12 da seção 5.2). O catálogo é salvo em CSV na pasta `catalogo`, junto com o diagrama acima.

In [ ]:
# ------------------------------------------------------------
# 6.2.1 Catálogo de dados da camada ouro
# ------------------------------------------------------------
# Cada linha: [nome do atributo, tipo, descrição, domínio, obrigatoriedade,
#              arquivo de origem, transformação aplicada]
linhas_catalogo = [
    # ---------------- fato_entregas ----------------
    ['fato_entregas.id_pedido', 'Texto', 'Código único do pedido (hash anonimizado)',
     'Texto de 32 caracteres; não se repete', 'Obrigatório: chave primária, não admite nulo',
     'olist_orders_dataset.csv (order_id)', 'Renomeado (R1); só pedidos entregues (R6) e de um único vendedor (R7)'],
    ['fato_entregas.id_cliente', 'Texto', 'Código do cliente que fez o pedido',
     'Texto de 32 caracteres; deve existir em dim_cliente', 'Obrigatório: chave estrangeira, não admite nulo',
     'olist_orders_dataset.csv (customer_id)', 'Renomeado (R1)'],
    ['fato_entregas.id_vendedor', 'Texto', 'Código do vendedor que atendeu o pedido',
     'Texto de 32 caracteres; deve existir em dim_vendedor', 'Obrigatório: chave estrangeira, não admite nulo',
     'olist_order_items_dataset.csv (seller_id)', 'Renomeado (R1); vendedor único do pedido (R7)'],
    ['fato_entregas.id_tempo_compra', 'Numérico', 'Data da compra no formato AAAAMMDD',
     '20160904 a 20181017; deve existir em dim_tempo', 'Obrigatório: chave estrangeira, não admite nulo',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Data convertida para inteiro AAAAMMDD'],
    ['fato_entregas.qtd_itens', 'Numérico', 'Quantidade de itens do pedido',
     'Inteiro de 1 a 25', 'Obrigatório',
     'olist_order_items_dataset.csv (order_item_id)', 'Contagem de itens por pedido (R7)'],
    ['fato_entregas.valor_produtos', 'Numérico', 'Soma dos preços dos itens do pedido, em R$',
     'Maior que 0 e até R$ 15.000', 'Obrigatório',
     'olist_order_items_dataset.csv (price)', 'Soma por pedido (R7); 2 casas decimais'],
    ['fato_entregas.valor_frete', 'Numérico', 'Soma do frete dos itens do pedido, em R$',
     'De 0 (frete grátis) até R$ 2.000', 'Obrigatório',
     'olist_order_items_dataset.csv (freight_value)', 'Soma por pedido (R7); 2 casas decimais'],
    ['fato_entregas.dias_aprovacao', 'Numérico', 'Etapa 1: dias entre a compra e a aprovação do pagamento',
     '0 a 31 dias', 'Opcional: nulo = data de aprovação ausente ou fora de ordem',
     'olist_orders_dataset.csv (order_approved_at - order_purchase_timestamp)', 'Diferença em dias (R8); negativo vira nulo (R9)'],
    ['fato_entregas.dias_manuseio', 'Numérico', 'Etapa 2: dias entre a aprovação e a postagem na transportadora (manuseio pelo vendedor)',
     '0 a 130 dias', 'Opcional: nulo = data ausente ou postagem registrada antes da aprovação',
     'olist_orders_dataset.csv (order_delivered_carrier_date - order_approved_at)', 'Diferença em dias (R8); negativo vira nulo (R9)'],
    ['fato_entregas.dias_transporte', 'Numérico', 'Etapa 3: dias entre a postagem e a entrega ao cliente',
     '0 a 210 dias', 'Opcional: nulo = data ausente ou entrega registrada antes da postagem',
     'olist_orders_dataset.csv (order_delivered_customer_date - order_delivered_carrier_date)', 'Diferença em dias (R8); negativo vira nulo (R9)'],
    ['fato_entregas.dias_entrega_total', 'Numérico', 'Tempo total de entrega (lead time): da compra até a entrega',
     'Maior que 0 e até 210 dias', 'Obrigatório',
     'olist_orders_dataset.csv (order_delivered_customer_date - order_purchase_timestamp)', 'Diferença em dias (R8)'],
    ['fato_entregas.dias_prometidos', 'Numérico', 'Prazo prometido ao cliente na compra',
     'Maior que 0 e até 160 dias', 'Obrigatório',
     'olist_orders_dataset.csv (order_estimated_delivery_date - order_purchase_timestamp)', 'Diferença em dias (R8)'],
    ['fato_entregas.dias_atraso', 'Numérico', 'Dias entre a data estimada e a entrega; negativo = entregue antes do prazo',
     'Inteiro de -150 a +190 dias', 'Obrigatório',
     'olist_orders_dataset.csv (order_delivered_customer_date, order_estimated_delivery_date)', 'Diferença entre as datas sem horário (R10)'],
    ['fato_entregas.flag_atraso', 'Booleano', '1 = entregue depois da data estimada; 0 = no prazo',
     '0 ou 1', 'Obrigatório',
     'olist_orders_dataset.csv (order_delivered_customer_date, order_estimated_delivery_date)', 'Regra de atraso por dia (R10)'],
    ['fato_entregas.flag_postagem_atrasada', 'Booleano', '1 = vendedor postou depois do prazo limite de postagem',
     '0 ou 1', 'Opcional: nulo = pedido sem data de postagem',
     'olist_orders_dataset.csv e olist_order_items_dataset.csv (shipping_limit_date)', 'Comparação entre postagem e prazo limite (R11)'],
    ['fato_entregas.nota_avaliacao', 'Numérico', 'Nota dada pelo cliente ao pedido',
     'Inteiro de 1 a 5', 'Opcional: nulo = pedido sem avaliação',
     'olist_order_reviews_dataset.csv (review_score)', 'Avaliação de resposta mais recente (R5)'],
    ['fato_entregas.tipo_envio', 'Categórico', 'Indica se o vendedor está na mesma UF do cliente',
     'Local, Interestadual', 'Obrigatório',
     'olist_customers_dataset.csv e olist_sellers_dataset.csv (customer_state, seller_state)', 'Comparação das UFs de origem e destino (R12)'],
    ['fato_entregas.escopo_envio', 'Categórico', 'Escopo geográfico do envio',
     'Mesma UF; Outra UF da mesma região; Outra região', 'Obrigatório',
     'olist_customers_dataset.csv e olist_sellers_dataset.csv (customer_state, seller_state)', 'Comparação de UF e região IBGE (R3 e R12)'],
    # ---------------- dim_cliente ----------------
    ['dim_cliente.id_cliente', 'Texto', 'Código do cliente no pedido (hash anonimizado)',
     'Texto de 32 caracteres; não se repete', 'Obrigatório: chave primária, não admite nulo',
     'olist_customers_dataset.csv (customer_id)', 'Renomeado (R1)'],
    ['dim_cliente.cep_prefixo', 'Texto', 'Cinco primeiros dígitos do CEP do cliente',
     'Texto com 5 dígitos, de 01000 a 99999', 'Obrigatório',
     'olist_customers_dataset.csv (customer_zip_code_prefix)', 'Número convertido em texto com zero à esquerda (R2)'],
    ['dim_cliente.cidade', 'Texto', 'Cidade do cliente (minúsculas, sem acento, como na origem)',
     'Nome de município brasileiro', 'Obrigatório',
     'olist_customers_dataset.csv (customer_city)', 'Renomeado (R1)'],
    ['dim_cliente.uf', 'Categórico', 'Unidade da Federação do cliente',
     'As 27 siglas de UF (AC a TO)', 'Obrigatório',
     'olist_customers_dataset.csv (customer_state)', 'Renomeado (R1)'],
    ['dim_cliente.regiao', 'Categórico', 'Região geográfica do cliente (IBGE)',
     'Norte, Nordeste, Centro-Oeste, Sudeste, Sul', 'Obrigatório',
     'olist_customers_dataset.csv (customer_state)', 'Derivada da UF por dicionário (R3)'],
    # ---------------- dim_vendedor ----------------
    ['dim_vendedor.id_vendedor', 'Texto', 'Código do vendedor (hash anonimizado)',
     'Texto de 32 caracteres; não se repete', 'Obrigatório: chave primária, não admite nulo',
     'olist_sellers_dataset.csv (seller_id)', 'Renomeado (R1)'],
    ['dim_vendedor.cep_prefixo', 'Texto', 'Cinco primeiros dígitos do CEP do vendedor',
     'Texto com 5 dígitos, de 01000 a 99999', 'Obrigatório',
     'olist_sellers_dataset.csv (seller_zip_code_prefix)', 'Número convertido em texto com zero à esquerda (R2)'],
    ['dim_vendedor.cidade', 'Texto', 'Cidade do vendedor, como digitada no cadastro',
     'Nome de município brasileiro (18 registros fora do padrão, ver seção 7.1)', 'Obrigatório',
     'olist_sellers_dataset.csv (seller_city)', 'Renomeado (R1); mantido sem correção (não é usado nas perguntas)'],
    ['dim_vendedor.uf', 'Categórico', 'Unidade da Federação do vendedor',
     'As 27 siglas de UF (AC a TO)', 'Obrigatório',
     'olist_sellers_dataset.csv (seller_state)', 'Renomeado (R1)'],
    ['dim_vendedor.regiao', 'Categórico', 'Região geográfica do vendedor (IBGE)',
     'Norte, Nordeste, Centro-Oeste, Sudeste, Sul', 'Obrigatório',
     'olist_sellers_dataset.csv (seller_state)', 'Derivada da UF por dicionário (R3)'],
    # ---------------- dim_tempo ----------------
    ['dim_tempo.id_tempo', 'Numérico', 'Data no formato AAAAMMDD (chave do calendário)',
     '20160904 a 20181017; não se repete', 'Obrigatório: chave primária, não admite nulo',
     'olist_orders_dataset.csv (intervalo de order_purchase_timestamp)', 'Calendário diário gerado com pd.date_range'],
    ['dim_tempo.data', 'Data', 'Data do calendário (AAAA-MM-DD)',
     '2016-09-04 a 2018-10-17', 'Obrigatório',
     'olist_orders_dataset.csv (intervalo de order_purchase_timestamp)', 'Calendário diário gerado com pd.date_range'],
    ['dim_tempo.ano', 'Numérico', 'Ano da data', '2016 a 2018', 'Obrigatório',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Extraído da data'],
    ['dim_tempo.mes', 'Numérico', 'Mês da data', 'Inteiro de 1 a 12', 'Obrigatório',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Extraído da data'],
    ['dim_tempo.ano_mes', 'Categórico', 'Ano e mês (AAAA-MM), usado na análise mensal',
     '2016-09 a 2018-10', 'Obrigatório',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Extraído da data'],
    ['dim_tempo.trimestre', 'Numérico', 'Trimestre do ano', 'Inteiro de 1 a 4', 'Obrigatório',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Extraído da data'],
    ['dim_tempo.dia_semana', 'Categórico', 'Dia da semana da data',
     'segunda, terça, quarta, quinta, sexta, sábado, domingo', 'Obrigatório',
     'olist_orders_dataset.csv (order_purchase_timestamp)', 'Extraído da data e traduzido por dicionário']
]

catalogo = pd.DataFrame(linhas_catalogo, columns=['Nome do atributo', 'Tipo', 'Descrição', 'Domínio',
                                                  'Obrigatoriedade', 'arquivo_origem', 'transformacao'])

# Linhagem = arquivo de origem + data de extração (registrada na coleta) + transformação aplicada
catalogo['Linhagem'] = ('Origem: ' + catalogo['arquivo_origem']
                        + ' | Extraído em: ' + texto_data_coleta + ' (' + fonte_coleta + ')'
                        + ' | Transformação: ' + catalogo['transformacao'])

# Mantém exatamente as 6 colunas exigidas pelo edital (item 3.3)
catalogo = catalogo[['Nome do atributo', 'Tipo', 'Descrição', 'Domínio', 'Obrigatoriedade', 'Linhagem']]

# utf-8-sig: o arquivo abre com acentos corretos também no Excel
catalogo.to_csv(os.path.join(pasta_catalogo, 'catalogo_dados.csv'), index=False, encoding='utf-8-sig')

print('Atributos catalogados:', len(catalogo))
print('Colunas do catálogo  :', list(catalogo.columns))
catalogo

In [ ]:
# ------------------------------------------------------------
# 6.2.2 Conferência: todo atributo das tabelas da camada ouro está no catálogo?
# ------------------------------------------------------------
atributos_ouro = []
for nome_coluna in fato_ouro.columns:
    atributos_ouro.append('fato_entregas.' + nome_coluna)
for nome_coluna in dim_cliente_ouro.columns:
    atributos_ouro.append('dim_cliente.' + nome_coluna)
for nome_coluna in dim_vendedor_ouro.columns:
    atributos_ouro.append('dim_vendedor.' + nome_coluna)
for nome_coluna in dim_tempo_ouro.columns:
    atributos_ouro.append('dim_tempo.' + nome_coluna)

atributos_sem_catalogo = []
for atributo in atributos_ouro:
    if atributo not in list(catalogo['Nome do atributo']):
        atributos_sem_catalogo.append(atributo)

print('Atributos nas tabelas da camada ouro:', len(atributos_ouro))
print('Atributos sem descrição no catálogo :', len(atributos_sem_catalogo), atributos_sem_catalogo)

# O diagrama do modelo também fica salvo na pasta "catalogo" (pedido no item 5.1 do edital)
diagrama_texto = '''Esquema estrela - camada ouro (MVP Olist Entregas)

dim_tempo    (PK id_tempo)    1 ---- N  fato_entregas.id_tempo_compra
dim_cliente  (PK id_cliente)  1 ---- N  fato_entregas.id_cliente
dim_vendedor (PK id_vendedor) 1 ---- N  fato_entregas.id_vendedor

fato_entregas (grão: 1 pedido entregue de um único vendedor)
  PK id_pedido | FK id_cliente | FK id_vendedor | FK id_tempo_compra
  medidas: qtd_itens, valor_produtos, valor_frete, dias_aprovacao, dias_manuseio,
           dias_transporte, dias_entrega_total, dias_prometidos, dias_atraso,
           flag_atraso, flag_postagem_atrasada, nota_avaliacao, tipo_envio, escopo_envio
dim_cliente:  id_cliente, cep_prefixo, cidade, uf, regiao
dim_vendedor: id_vendedor, cep_prefixo, cidade, uf, regiao
dim_tempo:    id_tempo, data, ano, mes, ano_mes, trimestre, dia_semana
'''
arquivo_diagrama = open(os.path.join(pasta_catalogo, 'diagrama_esquema_estrela.txt'), 'w', encoding='utf-8')
arquivo_diagrama.write(diagrama_texto)
arquivo_diagrama.close()

print()
print('Arquivos na pasta catalogo:', sorted(os.listdir(pasta_catalogo)))

## 7. Análise

### 7.1 Qualidade dos dados

A análise de qualidade é feita **atributo por atributo** nas 5 tabelas brutas da camada bronze (o dado como foi recebido) e, no fim, repetida na camada ouro para mostrar que os problemas foram tratados. As dimensões examinadas são as do item 3.5 do edital:

| Dimensão | Pergunta que responde |
|---|---|
| Completude | Há valores nulos ou ausentes? |
| Unicidade | Há registros duplicados? |
| Consistência | Há contradições entre atributos (ex.: entrega antes da postagem)? |
| Conformidade | Os valores respeitam o domínio declarado (ex.: UF válida, nota de 1 a 5)? |
| Acurácia | Há valores implausíveis (ex.: entrega em mais de 60 dias)? |
| Atualidade | Qual período os dados cobrem? A cobertura é regular? |

#### 7.1.1 Perfil de cada atributo (completude e valores distintos)

Para cada coluna de cada tabela: tipo, quantidade e percentual de nulos, valores distintos, mínimo e máximo (para colunas numéricas). O perfil é salvo na pasta `catalogo`.

In [ ]:
tabelas_bronze = {
    'pedidos': df_pedidos,
    'itens': df_itens,
    'clientes': df_clientes,
    'vendedores': df_vendedores,
    'avaliacoes': df_avaliacoes
}

lista_perfis = []

for nome_tabela in tabelas_bronze:
    tabela = tabelas_bronze[nome_tabela]

    # Uma linha por atributo (coluna) da tabela
    perfil = pd.DataFrame()
    perfil['atributo'] = tabela.columns
    perfil['tabela'] = nome_tabela
    perfil['tipo'] = tabela.dtypes.astype(str).values
    perfil['linhas'] = len(tabela)
    perfil['nulos'] = tabela.isnull().sum().values
    perfil['pct_nulos'] = (100 * tabela.isnull().mean()).round(2).values
    perfil['distintos'] = tabela.nunique().values

    # Mínimo e máximo só fazem sentido para colunas numéricas
    perfil['minimo'] = perfil['atributo'].map(tabela.min(numeric_only=True))
    perfil['maximo'] = perfil['atributo'].map(tabela.max(numeric_only=True))

    lista_perfis.append(perfil)

df_perfil = pd.concat(lista_perfis, ignore_index=True)
df_perfil = df_perfil[['tabela', 'atributo', 'tipo', 'linhas', 'nulos', 'pct_nulos', 'distintos', 'minimo', 'maximo']]

df_perfil.to_csv(os.path.join(pasta_catalogo, 'perfil_qualidade_bronze.csv'), index=False, encoding='utf-8-sig')
df_perfil

**Leitura do perfil:** as tabelas de itens, clientes e vendedores não têm nulos. Os nulos aparecem em `pedidos` — datas de aprovação (160), de postagem (1.783) e de entrega (2.965), o que é esperado para pedidos cancelados ou ainda em trânsito — e em `avaliacoes`, no título (88,3%) e no comentário (58,7%), que são campos opcionais para o cliente. O perfil já antecipa três pontos detalhados nas próximas células: (a) o CEP foi lido como número (o menor valor é 1003, ou seja, perdeu o zero à esquerda); (b) `review_id` tem 98.410 valores distintos em 99.224 linhas, então há avaliações repetidas; (c) a data estimada de entrega tem só 459 valores distintos em 99.441 pedidos, sinal de que ela não tem horário.

#### 7.1.2 Completude

In [ ]:
sem_data_entrega = df_pedidos['order_delivered_customer_date'].isnull()
status_entregue = df_pedidos['order_status'] == 'delivered'
pedidos_sem_avaliacao = ~df_pedidos['order_id'].isin(df_avaliacoes['order_id'])

qtd_sem_entrega = sem_data_entrega.sum()
qtd_entregue_sem_data = (sem_data_entrega & status_entregue).sum()
qtd_sem_aprovacao = df_pedidos['order_approved_at'].isnull().sum()
qtd_sem_postagem = df_pedidos['order_delivered_carrier_date'].isnull().sum()
qtd_sem_avaliacao = pedidos_sem_avaliacao.sum()
pct_sem_comentario = 100 * df_avaliacoes['review_comment_message'].isnull().mean()

print('Pedidos por status:')
print(df_pedidos['order_status'].value_counts())
print()
print('Pedidos sem data de entrega ao cliente:', qtd_sem_entrega)
print('   dos quais com status "delivered"    :', qtd_entregue_sem_data)
print('Pedidos sem data de aprovação         :', qtd_sem_aprovacao)
print('Pedidos sem data de postagem          :', qtd_sem_postagem)
print('Pedidos sem nenhuma avaliação         :', qtd_sem_avaliacao)
print(f'Avaliações sem comentário escrito     : {pct_sem_comentario:.1f}%')

#### 7.1.3 Unicidade

In [ ]:
avaliacoes_por_pedido = df_avaliacoes['order_id'].value_counts()
qtd_pedidos_com_2_avaliacoes = (avaliacoes_por_pedido > 1).sum()
qtd_review_id_repetido = df_avaliacoes['review_id'].duplicated().sum()

print('PEDIDOS    - order_id repetido                    :', df_pedidos['order_id'].duplicated().sum())
print('ITENS      - par (order_id, order_item_id) repetido:', df_itens.duplicated(subset=['order_id', 'order_item_id']).sum())
print('CLIENTES   - customer_id repetido                 :', df_clientes['customer_id'].duplicated().sum())
print('VENDEDORES - seller_id repetido                   :', df_vendedores['seller_id'].duplicated().sum())
print('AVALIAÇÕES - review_id repetido                   :', qtd_review_id_repetido)
print('AVALIAÇÕES - pedidos com mais de uma avaliação    :', qtd_pedidos_com_2_avaliacoes)
print()
print('Observação: customer_id muda a cada pedido. A mesma pessoa (customer_unique_id) aparece em')
print('vários pedidos:', df_clientes['customer_unique_id'].nunique(), 'pessoas para', len(df_clientes), 'códigos de cliente.')

#### 7.1.4 Consistência (contradições entre atributos)

In [ ]:
# Ordem esperada das datas: compra <= aprovação <= postagem <= entrega
qtd_aprovacao_antes_compra = (df_pedidos['order_approved_at'] < df_pedidos['order_purchase_timestamp']).sum()
qtd_postagem_antes_aprovacao = (df_pedidos['order_delivered_carrier_date'] < df_pedidos['order_approved_at']).sum()
qtd_entrega_antes_postagem = (df_pedidos['order_delivered_customer_date'] < df_pedidos['order_delivered_carrier_date']).sum()

# Status x datas: pedido cancelado não deveria ter data de entrega
cancelado = df_pedidos['order_status'] == 'canceled'
qtd_cancelado_com_entrega = (cancelado & df_pedidos['order_delivered_customer_date'].notnull()).sum()

# Grão: quantos pedidos têm itens de mais de um vendedor?
vendedores_por_pedido = df_itens.groupby('order_id')['seller_id'].nunique()
qtd_pedidos_multivendedor = (vendedores_por_pedido > 1).sum()

# Prazo limite de postagem anterior à própria compra?
itens_com_data_compra = df_itens.merge(df_pedidos[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')
qtd_prazo_antes_compra = (itens_com_data_compra['shipping_limit_date'] < itens_com_data_compra['order_purchase_timestamp']).sum()

print('Aprovação registrada antes da compra       :', qtd_aprovacao_antes_compra)
print('Postagem registrada antes da aprovação     :', qtd_postagem_antes_aprovacao)
print('Entrega registrada antes da postagem       :', qtd_entrega_antes_postagem)
print('Pedido cancelado com data de entrega       :', qtd_cancelado_com_entrega)
print('Pedidos com itens de mais de um vendedor   :', qtd_pedidos_multivendedor)
print('Itens com prazo de postagem antes da compra:', qtd_prazo_antes_compra)

#### 7.1.5 Conformidade (aderência ao domínio)

In [ ]:
lista_ufs = list(regiao_por_uf.keys())    # as 27 siglas válidas

qtd_uf_cliente_invalida = (~df_clientes['customer_state'].isin(lista_ufs)).sum()
qtd_uf_vendedor_invalida = (~df_vendedores['seller_state'].isin(lista_ufs)).sum()
qtd_nota_fora_escala = (~df_avaliacoes['review_score'].between(1, 5)).sum()
qtd_cep_cliente_sem_zero = (df_clientes['customer_zip_code_prefix'] < 10000).sum()
qtd_cep_vendedor_sem_zero = (df_vendedores['seller_zip_code_prefix'] < 10000).sum()

# Cidade com número, "@", "/" ou "\" não é um nome de cidade válido
cidade_fora_padrao = df_vendedores['seller_city'].str.contains(r'[0-9@/\\]')
qtd_cidade_fora_padrao = cidade_fora_padrao.sum()

print('Clientes com UF fora das 27 siglas            :', qtd_uf_cliente_invalida)
print('Vendedores com UF fora das 27 siglas          :', qtd_uf_vendedor_invalida)
print('Notas fora da escala de 1 a 5                 :', qtd_nota_fora_escala)
print('Status de pedido encontrados                  :', sorted(df_pedidos['order_status'].unique()))
print('CEPs de clientes que perderam o zero à esquerda  :', qtd_cep_cliente_sem_zero)
print('CEPs de vendedores que perderam o zero à esquerda:', qtd_cep_vendedor_sem_zero)
print('Cidades de vendedor fora do padrão            :', qtd_cidade_fora_padrao)
print(df_vendedores.loc[cidade_fora_padrao, 'seller_city'].tolist())

#### 7.1.6 Acurácia (valores implausíveis)

In [ ]:
qtd_preco_invalido = (df_itens['price'] <= 0).sum()
qtd_frete_zero = (df_itens['freight_value'] == 0).sum()

dias_ate_entrega = (df_pedidos['order_delivered_customer_date'] - df_pedidos['order_purchase_timestamp']).dt.days
qtd_mais_de_60_dias = (dias_ate_entrega > 60).sum()

# A data estimada tem horário? (se tudo for 00:00, a comparação precisa ser feita por dia - regra R10)
hora_estimada = df_pedidos['order_estimated_delivery_date'].dt.hour
minuto_estimado = df_pedidos['order_estimated_delivery_date'].dt.minute
pct_estimada_sem_horario = 100 * ((hora_estimada == 0) & (minuto_estimado == 0)).mean()

print('Preço do item: mínimo R$', df_itens['price'].min(), '| máximo R$', df_itens['price'].max())
print('Itens com preço menor ou igual a zero:', qtd_preco_invalido)
print('Itens com frete igual a zero         :', qtd_frete_zero)
print('Pedidos entregues em mais de 60 dias :', qtd_mais_de_60_dias)
print('Maior tempo de entrega (dias)        :', dias_ate_entrega.max())
print(f'Datas estimadas sem horário (00:00)  : {pct_estimada_sem_horario:.1f}%')

#### 7.1.7 Atualidade (período coberto)

In [ ]:
mes_compra = df_pedidos['order_purchase_timestamp'].dt.strftime('%Y-%m')
pedidos_por_mes = mes_compra.value_counts().sort_index()

meses_com_poucos_pedidos = pedidos_por_mes[pedidos_por_mes < 500]

print('Primeira compra:', df_pedidos['order_purchase_timestamp'].min())
print('Última compra  :', df_pedidos['order_purchase_timestamp'].max())
print('Meses com menos de 500 pedidos:')
print(meses_com_poucos_pedidos)

# Meses com poucos pedidos em cinza; meses com volume regular em azul
cores_meses = np.where(pedidos_por_mes.values < 500, cor_cinza, cor_azul)

plt.figure(figsize=(12, 4.5))
plt.bar(pedidos_por_mes.index, pedidos_por_mes.values, color=cores_meses)
plt.xticks(rotation=90)
plt.ylabel('Pedidos (todos os status)')
plt.title('Fig. 1 – Pedidos por mês de compra (em cinza: meses com menos de 500 pedidos)')
plt.savefig(os.path.join(pasta_evidencias, 'fig01_pedidos_por_mes.png'), dpi=150, bbox_inches='tight')
plt.show()

#### 7.1.8 Resumo dos problemas encontrados e do tratamento adotado

A tabela é montada a partir das contagens calculadas nas células anteriores (nenhum número foi digitado à mão).

In [ ]:
resumo_qualidade = pd.DataFrame({
    'dimensão': ['Completude', 'Completude', 'Completude', 'Completude',
                 'Unicidade', 'Unicidade',
                 'Consistência', 'Consistência', 'Consistência', 'Consistência (grão)',
                 'Conformidade', 'Conformidade',
                 'Acurácia', 'Acurácia', 'Acurácia',
                 'Atualidade'],
    'atributo': ['order_delivered_customer_date', 'order_approved_at', 'order_id (avaliações)',
                 'review_comment_title / message',
                 'order_id (avaliações)', 'review_id',
                 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_status',
                 'seller_id (itens)',
                 'customer/seller_zip_code_prefix', 'seller_city',
                 'freight_value', 'order_delivered_customer_date', 'order_estimated_delivery_date',
                 'order_purchase_timestamp'],
    'problema': ['Pedido sem data de entrega (cancelado, em trânsito etc.)',
                 'Data de aprovação ausente',
                 'Pedido sem avaliação',
                 'Comentário em texto livre ausente (campo opcional)',
                 'Pedido com mais de uma avaliação',
                 'Mesma avaliação ligada a mais de um pedido',
                 'Postagem registrada antes da aprovação',
                 'Entrega registrada antes da postagem',
                 'Pedido cancelado com data de entrega',
                 'Pedido com itens de mais de um vendedor',
                 'CEP gravado como número (perdeu o zero à esquerda)',
                 'Cidade fora do padrão (número, e-mail, "cidade/uf")',
                 'Frete igual a zero',
                 'Entrega em mais de 60 dias',
                 'Data estimada sem horário (sempre 00:00)',
                 'Poucos pedidos em 2016 e em set-out/2018'],
    'ocorrências': [f'{qtd_sem_entrega} ({qtd_entregue_sem_data} com status delivered)',
                    str(qtd_sem_aprovacao),
                    str(qtd_sem_avaliacao),
                    f'{pct_sem_comentario:.1f}% das avaliações',
                    str(qtd_pedidos_com_2_avaliacoes),
                    str(qtd_review_id_repetido),
                    str(qtd_postagem_antes_aprovacao),
                    str(qtd_entrega_antes_postagem),
                    str(qtd_cancelado_com_entrega),
                    str(qtd_pedidos_multivendedor),
                    f'{qtd_cep_cliente_sem_zero} clientes e {qtd_cep_vendedor_sem_zero} vendedores',
                    str(qtd_cidade_fora_padrao),
                    f'{qtd_frete_zero} itens',
                    str(qtd_mais_de_60_dias),
                    f'{pct_estimada_sem_horario:.0f}% dos pedidos',
                    f'{len(meses_com_poucos_pedidos)} meses'],
    'tratamento (regra)': ['Fora da tabela fato (R6)',
                           'Etapa de aprovação fica nula; pedido mantido (R9)',
                           'Nota nula, sem imputação',
                           'Coluna descartada (R4 - LGPD)',
                           'Mantida a avaliação mais recente (R5)',
                           'review_id não vai para a camada ouro; a nota é ligada ao pedido',
                           'Etapa de manuseio anulada; pedido mantido (R9)',
                           'Etapa de transporte anulada; pedido mantido (R9)',
                           'Excluído pelo filtro de status (R6)',
                           'Excluído da fato (R7)',
                           'Texto com 5 dígitos (R2)',
                           'Mantida: atributo descritivo, não usado nas perguntas',
                           'Mantido: frete grátis é um valor real',
                           'Mantida: atraso real e extremo',
                           'Atraso comparado por dia (R10)',
                           'P2 restrita a jan/2017-ago/2018'],
    'efeito nas perguntas': ['Só pedidos entregues têm atraso mensurável',
                             'P3 usa só pedidos com as 3 etapas válidas',
                             'P5 usa só pedidos avaliados',
                             'Nenhum',
                             'Evita contar o mesmo pedido duas vezes na P5',
                             'Nenhum',
                             'P3', 'P3', 'Nenhum',
                             'O atraso fica atribuível a um único vendedor (P3)',
                             'Chave de CEP correta nas dimensões',
                             'Nenhum',
                             'P4: frete médio um pouco menor',
                             'Puxam as médias de tempo para cima',
                             'Evita superestimar a taxa de atraso',
                             'Limitação registrada na autoavaliação']
})
resumo_qualidade.to_csv(os.path.join(pasta_catalogo, 'resumo_qualidade.csv'), index=False, encoding='utf-8-sig')
resumo_qualidade

#### 7.1.9 Verificação depois do tratamento (camada ouro × domínios do catálogo)

As tabelas relidas da camada ouro são confrontadas com os domínios declarados no catálogo de dados (seção 6.2). O resultado esperado é zero valores fora do domínio em todos os atributos.

In [ ]:
# Completude da fato depois do tratamento (nulos que restaram e por quê estão no catálogo)
print('Nulos que restaram na fato_entregas:')
nulos_fato = fato_ouro.isnull().sum()
print(nulos_fato[nulos_fato > 0])
print()

# Conformidade com os domínios do catálogo (os nulos permitidos são ignorados com dropna)
checagens = pd.DataFrame({
    'atributo': ['fato_entregas.qtd_itens', 'fato_entregas.valor_produtos', 'fato_entregas.valor_frete',
                 'fato_entregas.dias_aprovacao', 'fato_entregas.dias_manuseio', 'fato_entregas.dias_transporte',
                 'fato_entregas.dias_entrega_total', 'fato_entregas.dias_prometidos', 'fato_entregas.dias_atraso',
                 'fato_entregas.flag_atraso', 'fato_entregas.flag_postagem_atrasada', 'fato_entregas.nota_avaliacao',
                 'fato_entregas.tipo_envio', 'fato_entregas.escopo_envio',
                 'dim_cliente.uf', 'dim_vendedor.uf', 'dim_cliente.cep_prefixo', 'dim_vendedor.cep_prefixo',
                 'fato_entregas.id_pedido'],
    'domínio no catálogo': ['1 a 25', '0 a 15.000', '0 a 2.000',
                            '0 a 31', '0 a 130', '0 a 210',
                            '0 a 210', '0 a 160', '-150 a 190',
                            '0 ou 1', '0 ou 1', '1 a 5',
                            'Local, Interestadual', '3 categorias',
                            '27 UFs', '27 UFs', '5 dígitos', '5 dígitos',
                            'sem repetição'],
    'valores fora do domínio': [
        (~fato_ouro['qtd_itens'].between(1, 25)).sum(),
        (~fato_ouro['valor_produtos'].between(0.01, 15000)).sum(),
        (~fato_ouro['valor_frete'].between(0, 2000)).sum(),
        (~fato_ouro['dias_aprovacao'].dropna().between(0, 31)).sum(),
        (~fato_ouro['dias_manuseio'].dropna().between(0, 130)).sum(),
        (~fato_ouro['dias_transporte'].dropna().between(0, 210)).sum(),
        (~fato_ouro['dias_entrega_total'].between(0, 210)).sum(),
        (~fato_ouro['dias_prometidos'].between(0, 160)).sum(),
        (~fato_ouro['dias_atraso'].between(-150, 190)).sum(),
        (~fato_ouro['flag_atraso'].isin([0, 1])).sum(),
        (~fato_ouro['flag_postagem_atrasada'].dropna().isin([0, 1])).sum(),
        (~fato_ouro['nota_avaliacao'].dropna().between(1, 5)).sum(),
        (~fato_ouro['tipo_envio'].isin(['Local', 'Interestadual'])).sum(),
        (~fato_ouro['escopo_envio'].isin(['Mesma UF', 'Outra UF da mesma região', 'Outra região'])).sum(),
        (~dim_cliente_ouro['uf'].isin(lista_ufs)).sum(),
        (~dim_vendedor_ouro['uf'].isin(lista_ufs)).sum(),
        (dim_cliente_ouro['cep_prefixo'].str.len() != 5).sum(),
        (dim_vendedor_ouro['cep_prefixo'].str.len() != 5).sum(),
        fato_ouro['id_pedido'].duplicated().sum()
    ]
})
checagens['situação'] = np.where(checagens['valores fora do domínio'] == 0, 'CONFORME', 'NÃO CONFORME')
checagens

**Conclusão da análise de qualidade.** A base é de boa qualidade para o objetivo do MVP: não há chaves repetidas nas tabelas de pedidos, itens, clientes e vendedores, todas as UFs e notas estão no domínio e todos os preços são positivos. Os problemas encontrados são de três tipos:

1. **Ausências esperadas pelo próprio processo** (pedidos não entregues, avaliações sem comentário): tratadas pelo filtro de escopo (R6) ou mantidas como nulo, sem imputação.
2. **Erros de registro** (1.359 postagens antes da aprovação, 23 entregas antes da postagem, 6 pedidos cancelados com data de entrega, 18 cidades fora do padrão): tratados sem descartar o pedido inteiro; só a etapa afetada fica nula (R9).
3. **Pontos que exigem decisão de modelagem** (pedidos com mais de um vendedor, data estimada sem horário, CEP sem o zero à esquerda): resolvidos pelas regras R7, R10 e R2.

Depois do tratamento, os 19 testes de domínio da camada ouro deram **CONFORME**, e os nulos que restaram na fato são exatamente os declarados como opcionais no catálogo. Nenhum problema impede responder às perguntas; os que afetam alguma resposta são lembrados na discussão correspondente.

### 7.2 Solução do problema

Todas as respostas são calculadas sobre as tabelas da camada ouro **relidas do disco** (seção 5.5), unidas pelas chaves do esquema estrela. Só são usados comandos diretos do pandas (`merge`, `groupby`, `agg`, `mean`, `median`, `value_counts`) e estatística descritiva: médias, taxas e proporções.

In [ ]:
# ------------------------------------------------------------
# Base de análise = fato + dimensões (junções pelas chaves do esquema estrela)
# ------------------------------------------------------------
base_analise = fato_ouro.merge(dim_cliente_ouro[['id_cliente', 'uf', 'regiao']], on='id_cliente', how='left')
base_analise = base_analise.rename(columns={'uf': 'uf_cliente', 'regiao': 'regiao_cliente'})

base_analise = base_analise.merge(dim_vendedor_ouro[['id_vendedor', 'uf', 'regiao']], on='id_vendedor', how='left')
base_analise = base_analise.rename(columns={'uf': 'uf_vendedor', 'regiao': 'regiao_vendedor'})

base_analise = base_analise.merge(dim_tempo_ouro[['id_tempo', 'ano_mes']],
                                  left_on='id_tempo_compra', right_on='id_tempo', how='left')

# Indicadores gerais do processo
taxa_nacional = 100 * base_analise['flag_atraso'].mean()
tempo_medio_entrega = base_analise['dias_entrega_total'].mean()
prazo_medio_prometido = base_analise['dias_prometidos'].mean()

print('Pedidos na base de análise:', len(base_analise))
print(f'Taxa de atraso nacional   : {taxa_nacional:.2f}%')
print(f'Tempo médio de entrega    : {tempo_medio_entrega:.1f} dias')
print(f'Prazo médio prometido     : {prazo_medio_prometido:.1f} dias')
print(f'Folga média (prometido - real): {prazo_medio_prometido - tempo_medio_entrega:.1f} dias')

#### P1 — Quais UFs de destino têm as maiores taxas de atraso e quais concentram a maior parte dos pedidos atrasados?

Duas medidas diferentes: a **taxa de atraso** da UF mede o *risco* de um pedido atrasar ali; a **participação nos atrasos** mede o *volume* de pedidos atrasados que a UF gera. As duas levam a prioridades diferentes.

In [ ]:
atraso_por_uf = base_analise.groupby('uf_cliente').agg(
    pedidos=('id_pedido', 'count'),
    pedidos_atrasados=('flag_atraso', 'sum'),
    taxa_atraso=('flag_atraso', 'mean'),
    dias_entrega_medio=('dias_entrega_total', 'mean'),
    dias_prometidos_medio=('dias_prometidos', 'mean')
)
atraso_por_uf['taxa_atraso'] = 100 * atraso_por_uf['taxa_atraso']

total_atrasados = atraso_por_uf['pedidos_atrasados'].sum()
atraso_por_uf['pct_dos_atrasos'] = 100 * atraso_por_uf['pedidos_atrasados'] / total_atrasados
atraso_por_uf['vezes_a_taxa_nacional'] = atraso_por_uf['taxa_atraso'] / taxa_nacional

atraso_por_uf = atraso_por_uf.sort_values('taxa_atraso', ascending=False)
atraso_por_uf.round(2)

In [ ]:
# Fig. 2 - risco: taxa de atraso por UF (em laranja, as UFs com pelo menos 2 vezes a taxa nacional)
cores_uf = np.where(atraso_por_uf['taxa_atraso'] >= 2 * taxa_nacional, cor_laranja, cor_azul)

plt.figure(figsize=(9, 8))
barras = plt.barh(atraso_por_uf.index, atraso_por_uf['taxa_atraso'], color=cores_uf)
plt.bar_label(barras, fmt='%.1f%%', padding=3, fontsize=9)
plt.axvline(taxa_nacional, color=cor_texto, linestyle='--', linewidth=1.2,
            label=f'Taxa nacional ({taxa_nacional:.1f}%)')
plt.axvline(2 * taxa_nacional, color=cor_laranja, linestyle=':', linewidth=1.5,
            label=f'2 vezes a taxa nacional ({2 * taxa_nacional:.1f}%)')
plt.gca().invert_yaxis()     # maior taxa no topo
plt.grid(False)              # tira a grade horizontal...
plt.grid(True, axis='x')     # ...e usa a grade vertical
plt.xlabel('Taxa de atraso (% dos pedidos entregues)')
plt.title('Fig. 2 – Taxa de atraso por UF de destino')
plt.legend(loc='lower right')
plt.savefig(os.path.join(pasta_evidencias, 'fig02_taxa_atraso_por_uf.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. 3 - volume: Pareto dos pedidos atrasados por UF
pareto_uf = atraso_por_uf.sort_values('pedidos_atrasados', ascending=False)
pareto_uf['pct_acumulado'] = pareto_uf['pct_dos_atrasos'].cumsum()

# Quantas UFs são necessárias para chegar a 80% dos atrasos?
qtd_ufs_80 = (pareto_uf['pct_acumulado'] < 80).sum() + 1
print('UFs necessárias para somar 80% dos pedidos atrasados:', qtd_ufs_80)
print(pareto_uf[['pedidos', 'taxa_atraso', 'pct_dos_atrasos', 'pct_acumulado']].head(qtd_ufs_80).round(1))

plt.figure(figsize=(11, 5))
plt.bar(pareto_uf.index, pareto_uf['pct_dos_atrasos'], color=cor_azul, label='% dos pedidos atrasados')
plt.plot(pareto_uf.index, pareto_uf['pct_acumulado'], color=cor_laranja, marker='o', markersize=4,
         label='% acumulado')
plt.axhline(80, color=cor_texto, linestyle='--', linewidth=1, label='80% dos atrasos')
plt.ylim(0, 105)
plt.ylabel('% do total de pedidos atrasados')
plt.title('Fig. 3 – Pareto dos pedidos atrasados por UF de destino')
plt.legend(loc='center right')
plt.savefig(os.path.join(pasta_evidencias, 'fig03_pareto_atrasos_por_uf.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P1 (complemento) - O prazo prometido cobre o tempo real de entrega de cada UF?
# Percentil 90 = tempo dentro do qual acontecem 90% das entregas daquela UF
p90_por_uf = base_analise.groupby('uf_cliente')['dias_entrega_total'].quantile(0.9)
atraso_por_uf['p90_dias_entrega'] = p90_por_uf
atraso_por_uf['promessa_cobre_p90'] = np.where(
    atraso_por_uf['dias_prometidos_medio'] >= atraso_por_uf['p90_dias_entrega'], 'Sim', 'Não')

# Taxa de atraso dos pedidos das UFs em que a promessa cobre (ou não) o P90
base_analise['promessa_cobre_p90'] = base_analise['uf_cliente'].map(atraso_por_uf['promessa_cobre_p90'])
cobertura_promessa = base_analise.groupby('promessa_cobre_p90').agg(
    ufs=('uf_cliente', 'nunique'),
    pedidos=('id_pedido', 'count'),
    taxa_atraso=('flag_atraso', 'mean')
)
cobertura_promessa['taxa_atraso'] = 100 * cobertura_promessa['taxa_atraso']

ufs_sem_cobertura = list(atraso_por_uf[atraso_por_uf['promessa_cobre_p90'] == 'Não'].index)
print('UFs em que o prazo médio prometido NÃO cobre o P90 do tempo real:', ufs_sem_cobertura)
print()
print(atraso_por_uf[['dias_entrega_medio', 'p90_dias_entrega', 'dias_prometidos_medio', 'promessa_cobre_p90']].round(1).head(10))
cobertura_promessa.round(2)

**Resposta à P1**

- **Risco:** cinco UFs do Nordeste têm taxa de atraso de pelo menos o dobro da nacional (6,85%): **AL (21,5%)**, MA (17,6%), SE (15,4%), PI (13,9%) e CE (13,9%). Em Alagoas, um pedido em cada cinco chega depois da data prometida.
- **Volume:** **SP e RJ somam 50,7% de todos os pedidos atrasados**, e 8 UFs concentram 80,5% (Fig. 3). SP tem taxa baixa (4,6%), mas é a UF com mais pedidos; o RJ combina volume alto (12.194 pedidos) e taxa alta (12,2%) e por isso gera, sozinho, 22,8% dos atrasos (1.490 pedidos).
- **Causa provável:** em 12 UFs o prazo médio prometido fica abaixo do percentil 90 do tempo real de entrega (ex.: em AL a promessa média é de 32,6 dias, mas 10% das entregas levam mais de 39,8 dias). Os pedidos dessas 12 UFs atrasam **12,0%** das vezes, contra **5,1%** nas 15 UFs em que a promessa cobre o P90.

**Discussão (Engenharia de Produção)**

Risco e volume levam a ações diferentes. Para as UFs de alto risco e pouco volume (AL, MA, SE, PI e CE, que juntas geram 7,7% dos atrasos), a ação de menor custo é **recalibrar o prazo prometido** pela distribuição real dos tempos de entrega (por exemplo, prometer o P90 da UF). É uma mudança de parâmetro no sistema, sem investimento físico. Para o RJ, que reúne risco e volume, a recalibração sozinha não basta: o volume justifica negociar nível de serviço com as transportadoras da rota ou avaliar estoque mais próximo do cliente. No país, a promessa média (23,7 dias) tem 11,1 dias de folga sobre o tempo médio real (12,6 dias), mas a folga é uma média: o atraso nasce na cauda da distribuição de algumas rotas, e não no tempo médio.

**Cuidados na interpretação:** RR (41 pedidos), AC e AP têm poucos pedidos, e suas taxas mudam muito com poucos casos. A comparação com o P90 é descritiva: mostra associação, não prova causa.

#### P2 — Como a taxa de atraso evoluiu mês a mês entre jan/2017 e ago/2018 e em quais meses houve picos?

O período é restrito a jan/2017–ago/2018 porque os meses das pontas têm poucos pedidos (seção 7.1.7) e gerariam taxas instáveis.

In [ ]:
filtro_periodo = (base_analise['ano_mes'] >= '2017-01') & (base_analise['ano_mes'] <= '2018-08')
base_periodo = base_analise[filtro_periodo]

atraso_por_mes = base_periodo.groupby('ano_mes').agg(
    pedidos=('id_pedido', 'count'),
    taxa_atraso=('flag_atraso', 'mean'),
    dias_entrega_medio=('dias_entrega_total', 'mean'),
    dias_prometidos_medio=('dias_prometidos', 'mean')
)
atraso_por_mes['taxa_atraso'] = 100 * atraso_por_mes['taxa_atraso']

mediana_mensal = atraso_por_mes['taxa_atraso'].median()
atraso_por_mes['vezes_a_mediana'] = atraso_por_mes['taxa_atraso'] / mediana_mensal
atraso_por_mes['pico'] = np.where(atraso_por_mes['vezes_a_mediana'] >= 2, 'Sim', 'Não')

print(f'Mediana das taxas mensais: {mediana_mensal:.2f}%')
print('Pedidos no período:', len(base_periodo))
atraso_por_mes.round(2)

In [ ]:
# Fig. 4 - taxa de atraso mês a mês, com a mediana e o limite de pico (2 vezes a mediana)
meses_pico = atraso_por_mes[atraso_por_mes['pico'] == 'Sim']

plt.figure(figsize=(11, 5))
plt.plot(atraso_por_mes.index, atraso_por_mes['taxa_atraso'], color=cor_azul, marker='o',
         label='Taxa de atraso do mês')
plt.scatter(meses_pico.index, meses_pico['taxa_atraso'], color=cor_laranja, s=90, zorder=3,
            label='Pico (2 vezes a mediana ou mais)')
plt.axhline(mediana_mensal, color=cor_texto, linestyle='--', linewidth=1,
            label=f'Mediana mensal ({mediana_mensal:.1f}%)')
plt.axhline(2 * mediana_mensal, color=cor_laranja, linestyle=':', linewidth=1.5,
            label=f'2 vezes a mediana ({2 * mediana_mensal:.1f}%)')
plt.xticks(rotation=45, ha='right')
plt.ylim(bottom=0)
plt.ylabel('Taxa de atraso (%)')
plt.title('Fig. 4 – Taxa de atraso por mês da compra (jan/2017 a ago/2018)')
plt.legend(loc='upper left')
plt.savefig(os.path.join(pasta_evidencias, 'fig04_taxa_atraso_por_mes.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. 5 - o prazo prometido acompanhou os meses de pico?
plt.figure(figsize=(11, 5))
plt.plot(atraso_por_mes.index, atraso_por_mes['dias_prometidos_medio'], color=cor_cinza, marker='s',
         linewidth=2.5, label='Prazo médio prometido')
plt.plot(atraso_por_mes.index, atraso_por_mes['dias_entrega_medio'], color=cor_azul, marker='o',
         label='Tempo médio real de entrega')
plt.xticks(rotation=45, ha='right')
plt.ylim(bottom=0)
plt.ylabel('Dias')
plt.title('Fig. 5 – Prazo prometido × tempo real de entrega, por mês da compra')
plt.legend(loc='lower left')
plt.savefig(os.path.join(pasta_evidencias, 'fig05_prometido_vs_real_por_mes.png'), dpi=150, bbox_inches='tight')
plt.show()

**Resposta à P2**

A mediana das taxas mensais foi de 4,5%. Três meses passaram do limite de pico (2 vezes a mediana, ou seja, 9,0%): **nov/2017 (12,5%)**, **fev/2018 (14,2%)** e **mar/2018 (19,2%, 4,3 vezes a mediana)**. Nos demais meses, a taxa ficou entre 1,2% (jun/2018) e 7,5% (dez/2017).

**Discussão (Engenharia de Produção)**

Os picos têm naturezas diferentes:

- **nov/2017 — pico de demanda:** é o mês da Black Friday e o de maior volume do período (7.195 pedidos, 63% acima de out/2017). O tempo médio real subiu de 11,9 para 15,2 dias. É o padrão clássico de **demanda acima da capacidade** logística instalada.
- **fev–mar/2018 — pico sem aumento de volume:** o volume ficou estável (de 6.500 a 6.900 pedidos), mas o tempo real subiu para 17,0 e 16,4 dias. A causa está do lado da oferta logística. Como a base não traz a transportadora nem eventos externos, a causa **não pode ser comprovada com estes dados**; uma hipótese compatível com o calendário é a greve dos funcionários dos Correios iniciada em 12/03/2018 (VERDÉLIO, 2018), somada ao Carnaval em fevereiro.

A Fig. 5 acrescenta o ponto mais importante para a gestão: **o prazo prometido não acompanhou a piora do tempo real**. Em mar/2018, o mês mais crítico, a promessa média até caiu (22,0 dias, contra 24,5 em fev/2018). Quando a promessa foi alongada, a taxa caiu (dez/2017: 27,7 dias e 7,5% de atraso; jun/2018: 27,7 dias e 1,2%); quando foi encurtada, a taxa subiu (ago/2018: 15,1 dias e 6,3%, contra 3,4% em jul/2018).

**Decisão sugerida:** planejar capacidade (transportadoras de reserva, reforço de equipe) e alongar temporariamente o prazo prometido em picos previsíveis, como a Black Friday; criar um alerta mensal quando a taxa passar de 2 vezes a mediana, para reagir a choques como o de fev–mar/2018.

**Cuidados na interpretação:** o mês é o da compra, não o da entrega; um pedido de fim de mês pode ter sido entregue no mês seguinte. Os meses de 2016 e de set–out/2018 ficaram de fora por terem poucos pedidos.

#### P3 — Em qual etapa do processo está o gargalo do tempo de entrega, e quanto a postagem fora do prazo pelo vendedor aumenta a taxa de atraso?

O tempo total de entrega é decomposto nas três etapas do processo: **aprovação do pagamento** (compra → aprovação), **manuseio pelo vendedor** (aprovação → postagem na transportadora) e **transporte** (postagem → entrega). Entram só os pedidos com as três etapas válidas (regra R9).

In [ ]:
etapas = ['dias_aprovacao', 'dias_manuseio', 'dias_transporte']
nomes_etapas = ['Aprovação do pagamento', 'Manuseio pelo vendedor', 'Transporte']

base_etapas = base_analise.dropna(subset=etapas)
print('Pedidos com as 3 etapas válidas:', len(base_etapas), 'de', len(base_analise))

# Tempo médio de cada etapa e sua participação no tempo total
media_etapas = base_etapas[etapas].mean()
participacao_etapas = 100 * media_etapas / media_etapas.sum()

# Mesmas médias, separando pedidos no prazo (0) e atrasados (1)
etapas_por_situacao = base_etapas.groupby('flag_atraso')[etapas].mean()
etapas_por_situacao.index = ['No prazo', 'Atrasado']
aumento_etapas = etapas_por_situacao.loc['Atrasado'] - etapas_por_situacao.loc['No prazo']

resumo_etapas = pd.DataFrame({
    'etapa': nomes_etapas,
    'media_geral_dias': media_etapas.values,
    'participacao_no_total_pct': participacao_etapas.values,
    'media_no_prazo_dias': etapas_por_situacao.loc['No prazo'].values,
    'media_atrasados_dias': etapas_por_situacao.loc['Atrasado'].values,
    'aumento_nos_atrasados_dias': aumento_etapas.values
})
resumo_etapas['pct_do_aumento'] = 100 * resumo_etapas['aumento_nos_atrasados_dias'] / aumento_etapas.sum()
resumo_etapas.round(2)

In [ ]:
# Fig. 6 - tempo médio de cada etapa: pedidos no prazo x atrasados
posicoes = np.arange(len(etapas))     # 0, 1, 2 (uma posição por etapa)
largura = 0.38

plt.figure(figsize=(9, 5))
barras_prazo = plt.bar(posicoes - largura / 2, etapas_por_situacao.loc['No prazo'], width=largura,
                       color=cor_azul, label='Entregue no prazo')
barras_atraso = plt.bar(posicoes + largura / 2, etapas_por_situacao.loc['Atrasado'], width=largura,
                        color=cor_laranja, label='Entregue com atraso')
plt.bar_label(barras_prazo, fmt='%.1f', padding=2)
plt.bar_label(barras_atraso, fmt='%.1f', padding=2)
plt.xticks(posicoes, nomes_etapas)
plt.margins(y=0.1)    # espaço para os rótulos em cima das barras
plt.ylabel('Tempo médio da etapa (dias)')
plt.title('Fig. 6 – Tempo médio de cada etapa: pedidos no prazo × atrasados')
plt.legend()
plt.savefig(os.path.join(pasta_evidencias, 'fig06_etapas_prazo_vs_atraso.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P3 (complemento) - postagem fora do prazo pelo vendedor (regra R11)
base_postagem = base_analise.dropna(subset=['flag_postagem_atrasada'])

atraso_por_postagem = base_postagem.groupby('flag_postagem_atrasada').agg(
    pedidos=('id_pedido', 'count'),
    taxa_atraso=('flag_atraso', 'mean'),
    dias_manuseio_medio=('dias_manuseio', 'mean')
)
atraso_por_postagem['taxa_atraso'] = 100 * atraso_por_postagem['taxa_atraso']
atraso_por_postagem['pct_dos_pedidos'] = 100 * atraso_por_postagem['pedidos'] / atraso_por_postagem['pedidos'].sum()
atraso_por_postagem.index = ['Postado no prazo', 'Postado com atraso']

razao_postagem = (atraso_por_postagem.loc['Postado com atraso', 'taxa_atraso']
                  / atraso_por_postagem.loc['Postado no prazo', 'taxa_atraso'])

# Dos pedidos que atrasaram, quantos tinham sido postados com atraso pelo vendedor?
pedidos_atrasados = base_postagem[base_postagem['flag_atraso'] == 1]
pct_atrasos_com_postagem_atrasada = 100 * pedidos_atrasados['flag_postagem_atrasada'].mean()

print(f'A taxa de atraso é {razao_postagem:.1f} vezes maior quando o vendedor posta fora do prazo.')
print(f'Dos pedidos atrasados, {pct_atrasos_com_postagem_atrasada:.1f}% foram postados com atraso pelo vendedor.')
atraso_por_postagem.round(2)

In [ ]:
# Fig. 7 - taxa de atraso conforme a postagem do vendedor
plt.figure(figsize=(7, 5))
barras = plt.bar(atraso_por_postagem.index, atraso_por_postagem['taxa_atraso'], color=[cor_azul, cor_laranja],
                 width=0.55)
plt.bar_label(barras, fmt='%.1f%%', padding=3)
plt.margins(y=0.1)
plt.axhline(taxa_nacional, color=cor_texto, linestyle='--', linewidth=1,
            label=f'Taxa nacional ({taxa_nacional:.1f}%)')
plt.ylabel('Taxa de atraso na entrega (%)')
plt.title('Fig. 7 – Taxa de atraso conforme a postagem do vendedor')
plt.legend()
plt.savefig(os.path.join(pasta_evidencias, 'fig07_atraso_por_postagem.png'), dpi=150, bbox_inches='tight')
plt.show()

**Resposta à P3**

O **transporte é o gargalo**. Ele responde por **74,2%** do tempo médio de entrega (9,4 dias), contra 22,6% do manuseio pelo vendedor (2,9 dias) e 3,2% da aprovação do pagamento (0,4 dia). Nos pedidos atrasados, o transporte passa de 8,0 para **27,9 dias**: 86,8% do tempo a mais dos pedidos atrasados está nessa etapa (Fig. 6). A postagem fora do prazo pelo vendedor também pesa: 9,1% dos pedidos foram postados com atraso, e eles atrasam na entrega **20,9%** das vezes, contra 5,4% dos postados no prazo, ou seja, **3,8 vezes mais** (Fig. 7).

**Discussão (Engenharia de Produção)**

Pela Teoria das Restrições, a melhoria deve começar pela restrição do sistema, e aqui ela está fora do controle direto do vendedor: **72,2% dos pedidos atrasados foram postados no prazo** e atrasaram depois, já com a transportadora. Isso aponta para duas frentes com pesos diferentes:

1. **Transportadoras (frente principal):** acordo de nível de serviço (SLA) por rota, com o tempo de transporte acompanhado como indicador, e redistribuição de volume nas rotas de pior desempenho (a P1 e a P4 mostram quais são).
2. **Vendedores (frente secundária, mas barata):** a postagem atrasada funciona como **alerta antecipado**. Quando o vendedor posta fora do prazo, o risco de atraso quase quadruplica; o *marketplace* pode monitorar esse indicador e agir (avisar o cliente, priorizar o envio, cobrar o vendedor) antes de o atraso acontecer.

**Cuidados na interpretação:** a etapa de manuseio vai até a data em que a transportadora registrou a coleta, então parte da espera pela coleta pode estar contada como manuseio do vendedor. Ficaram de fora 1.336 pedidos com alguma etapa nula (datas ausentes ou fora de ordem, regra R9).

#### P4 — Qual a relação entre o escopo geográfico do envio (entrega na mesma UF vs. interestadual e entre regiões) com o tempo de entrega, o valor do frete e a taxa de atraso?

A comparação usa as colunas criadas pela regra R12: `tipo_envio` (Local × Interestadual) e `escopo_envio` (Mesma UF × Outra UF da mesma região × Outra região).

In [ ]:
# Local x Interestadual (coluna tipo_envio)
resumo_tipo = base_analise.groupby('tipo_envio').agg(
    pedidos=('id_pedido', 'count'),
    dias_entrega_medio=('dias_entrega_total', 'mean'),
    dias_prometidos_medio=('dias_prometidos', 'mean'),
    frete_medio=('valor_frete', 'mean'),
    taxa_atraso=('flag_atraso', 'mean')
)
resumo_tipo = resumo_tipo.loc[['Local', 'Interestadual']]    # ordem lógica
resumo_tipo['taxa_atraso'] = 100 * resumo_tipo['taxa_atraso']
resumo_tipo['pct_dos_pedidos'] = 100 * resumo_tipo['pedidos'] / resumo_tipo['pedidos'].sum()

razao_interestadual = resumo_tipo.loc['Interestadual', 'taxa_atraso'] / resumo_tipo.loc['Local', 'taxa_atraso']
print(f'Taxa de atraso interestadual / local = {razao_interestadual:.2f} vezes')
display(resumo_tipo.round(2))

# Os três escopos (coluna escopo_envio)
ordem_escopo = ['Mesma UF', 'Outra UF da mesma região', 'Outra região']
resumo_escopo = base_analise.groupby('escopo_envio').agg(
    pedidos=('id_pedido', 'count'),
    dias_entrega_medio=('dias_entrega_total', 'mean'),
    dias_prometidos_medio=('dias_prometidos', 'mean'),
    frete_medio=('valor_frete', 'mean'),
    taxa_atraso=('flag_atraso', 'mean')
)
resumo_escopo = resumo_escopo.loc[ordem_escopo]
resumo_escopo['taxa_atraso'] = 100 * resumo_escopo['taxa_atraso']
resumo_escopo['pct_dos_pedidos'] = 100 * resumo_escopo['pedidos'] / resumo_escopo['pedidos'].sum()
resumo_escopo['folga_media_dias'] = resumo_escopo['dias_prometidos_medio'] - resumo_escopo['dias_entrega_medio']
display(resumo_escopo.round(2))

In [ ]:
# Fig. 8 - tempo, frete e atraso por escopo do envio (três gráficos lado a lado)
rotulos_escopo = ['Mesma UF', 'Outra UF da\nmesma região', 'Outra\nregião']

plt.figure(figsize=(14, 4.8))

plt.subplot(1, 3, 1)
barras = plt.bar(rotulos_escopo, resumo_escopo['dias_entrega_medio'], color=cor_azul)
plt.bar_label(barras, fmt='%.1f', padding=2)
plt.margins(y=0.12)
plt.ylabel('Dias')
plt.title('Tempo médio de entrega')

plt.subplot(1, 3, 2)
barras = plt.bar(rotulos_escopo, resumo_escopo['frete_medio'], color=cor_azul)
plt.bar_label(barras, fmt='R$ %.2f', padding=2)
plt.margins(y=0.12)
plt.ylabel('R$ por pedido')
plt.title('Frete médio')

plt.subplot(1, 3, 3)
barras = plt.bar(rotulos_escopo, resumo_escopo['taxa_atraso'], color=cor_laranja)
plt.bar_label(barras, fmt='%.1f%%', padding=2)
plt.margins(y=0.12)
plt.ylabel('% dos pedidos')
plt.title('Taxa de atraso')

plt.suptitle('Fig. 8 – Escopo geográfico do envio: tempo, frete e atraso', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(pasta_evidencias, 'fig08_escopo_do_envio.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P4 (complemento) - de onde saem os pedidos e para onde vão os envios "Outra UF da mesma região"?
origem_dos_pedidos = 100 * base_analise['uf_vendedor'].value_counts(normalize=True)
print('UF do vendedor (% dos pedidos):')
print(origem_dos_pedidos.head(5).round(1))
print()

envios_mesma_regiao = base_analise[base_analise['escopo_envio'] == 'Outra UF da mesma região']
destino_mesma_regiao = 100 * envios_mesma_regiao['uf_cliente'].value_counts(normalize=True)
print('Destino dos envios "Outra UF da mesma região" (% desses envios):')
print(destino_mesma_regiao.head(4).round(1))
print()

# Mesmo comparando só vendedores de SP, o envio interestadual atrasa mais?
base_vendedores_sp = base_analise[base_analise['uf_vendedor'] == 'SP']
resumo_vendedores_sp = base_vendedores_sp.groupby('tipo_envio').agg(
    pedidos=('id_pedido', 'count'),
    dias_entrega_medio=('dias_entrega_total', 'mean'),
    frete_medio=('valor_frete', 'mean'),
    taxa_atraso=('flag_atraso', 'mean')
)
resumo_vendedores_sp['taxa_atraso'] = 100 * resumo_vendedores_sp['taxa_atraso']
print('Só vendedores de SP:')
resumo_vendedores_sp.round(2)

**Resposta à P4**

O escopo geográfico do envio afeta os três indicadores:

- **Local × interestadual:** o envio interestadual (64,0% dos pedidos) leva em média **15,2 dias**, contra **8,0** do envio local; custa **R$ 26,61** de frete, contra **R$ 15,23** (75% a mais); e atrasa **8,1%** das vezes, contra **4,6%** (1,79 vez a taxa do envio local).
- **Três escopos:** o tempo e o frete crescem junto com o escopo (8,0 → 13,9 → 16,1 dias; R$ 15,23 → R$ 23,34 → R$ 28,86). A taxa de atraso, não: ela é **maior para outra UF da mesma região (8,7%) do que para outra região (7,8%)** (Fig. 8).

**Discussão (Engenharia de Produção)**

O resultado da taxa de atraso mostra que a distância, sozinha, não explica o atraso. Duas leituras se completam:

1. **Calibração da promessa:** o prazo prometido cresce com o escopo, e a folga média (prometido − real) é maior para outra região (12,5 dias) do que para outra UF da mesma região (11,2 dias). As rotas longas recebem mais folga; as rotas médias, menos.
2. **Destino:** 37,0% dos envios para outra UF da mesma região vão para o RJ, uma das UFs com maior taxa de atraso do país (12,2%, P1). Como 70,9% dos pedidos saem de vendedores de SP, a rota SP→RJ pesa muito nesse grupo.

Olhando só os vendedores de SP (o que retira o efeito da mistura de origens), o envio interestadual atrasa 9,6% das vezes, contra 4,7% do envio local (2,05 vezes). Para a gestão, isso sugere: (a) prazo e frete definidos **por rota**, e não só pela distância; (b) incentivo a vendedores e estoques fora de SP, perto dos clientes de maior risco, o que reduziria ao mesmo tempo frete, tempo de entrega e atraso.

**Cuidados na interpretação:** o escopo por UF é uma aproximação da distância (um envio dentro de MG ou do PA pode ser mais longo que um envio SP→RJ). O frete médio também depende do peso e do tamanho do produto, que não entraram neste MVP.

#### P5 — Quanto o atraso reduz a nota de avaliação do cliente?

Entram só os pedidos com avaliação. Além da nota média, são comparadas a proporção de **notas ruins** (1 ou 2) e a proporção de **notas máximas** (5).

In [ ]:
base_avaliada = base_analise.dropna(subset=['nota_avaliacao']).copy()
base_avaliada['nota_ruim'] = np.where(base_avaliada['nota_avaliacao'] <= 2, 1, 0)
base_avaliada['nota_maxima'] = np.where(base_avaliada['nota_avaliacao'] == 5, 1, 0)

nota_por_situacao = base_avaliada.groupby('flag_atraso').agg(
    pedidos=('id_pedido', 'count'),
    nota_media=('nota_avaliacao', 'mean'),
    nota_mediana=('nota_avaliacao', 'median'),
    pct_nota_ruim=('nota_ruim', 'mean'),
    pct_nota_5=('nota_maxima', 'mean')
)
nota_por_situacao['pct_nota_ruim'] = 100 * nota_por_situacao['pct_nota_ruim']
nota_por_situacao['pct_nota_5'] = 100 * nota_por_situacao['pct_nota_5']
nota_por_situacao.index = ['No prazo', 'Atrasado']

diferenca_nota = nota_por_situacao.loc['No prazo', 'nota_media'] - nota_por_situacao.loc['Atrasado', 'nota_media']
razao_nota_ruim = nota_por_situacao.loc['Atrasado', 'pct_nota_ruim'] / nota_por_situacao.loc['No prazo', 'pct_nota_ruim']

# Que parte de todas as notas ruins vem de pedidos atrasados?
notas_ruins = base_avaliada[base_avaliada['nota_ruim'] == 1]
pct_notas_ruins_de_atrasos = 100 * notas_ruins['flag_atraso'].mean()
pct_pedidos_atrasados = 100 * base_avaliada['flag_atraso'].mean()

print(f'Diferença de nota média (no prazo - atrasado): {diferenca_nota:.2f} ponto')
print(f'A proporção de notas ruins é {razao_nota_ruim:.1f} vezes maior nos pedidos atrasados.')
print(f'Os atrasados são {pct_pedidos_atrasados:.1f}% dos pedidos avaliados, mas geram {pct_notas_ruins_de_atrasos:.1f}% das notas ruins.')
nota_por_situacao.round(2)

In [ ]:
# P5 (complemento) - nota média por faixa de antecedência ou de atraso (dias_atraso)
# dias_atraso negativo = entregue antes da data prometida; 0 = no próprio dia
limites_faixas = [-1000, -21, -11, -4, 0, 3, 7, 14, 1000]
nomes_faixas = ['21+ dias antes', '11 a 20 dias antes', '4 a 10 dias antes', '0 a 3 dias antes',
                '1 a 3 dias depois', '4 a 7 dias depois', '8 a 14 dias depois', '15+ dias depois']
base_avaliada['faixa_atraso'] = pd.cut(base_avaliada['dias_atraso'], bins=limites_faixas, labels=nomes_faixas)

nota_por_faixa = base_avaliada.groupby('faixa_atraso', observed=True).agg(
    pedidos=('id_pedido', 'count'),
    nota_media=('nota_avaliacao', 'mean'),
    pct_nota_ruim=('nota_ruim', 'mean')
)
nota_por_faixa['pct_nota_ruim'] = 100 * nota_por_faixa['pct_nota_ruim']
nota_por_faixa.round(2)

In [ ]:
# Fig. 9 - nota média por faixa (azul = no prazo, laranja = atrasado)
rotulos_faixas = ['21+\nantes', '11 a 20\nantes', '4 a 10\nantes', '0 a 3\nantes',
                  '1 a 3\ndepois', '4 a 7\ndepois', '8 a 14\ndepois', '15+\ndepois']
cores_faixas = [cor_azul, cor_azul, cor_azul, cor_azul, cor_laranja, cor_laranja, cor_laranja, cor_laranja]

plt.figure(figsize=(11, 5))
barras = plt.bar(rotulos_faixas, nota_por_faixa['nota_media'], color=cores_faixas)
plt.bar_label(barras, fmt='%.2f', padding=2)
plt.ylim(0, 5)
plt.xlabel('Dias em relação à data prometida (antes = antecipado; depois = atrasado)')
plt.ylabel('Nota média (1 a 5)')
plt.title('Fig. 9 – Nota média da avaliação por faixa de antecedência ou de atraso')
plt.savefig(os.path.join(pasta_evidencias, 'fig09_nota_por_faixa_de_atraso.png'), dpi=150, bbox_inches='tight')
plt.show()

**Resposta à P5**

O atraso reduz a nota média em **2,04 pontos**: 4,31 nos pedidos entregues no prazo, contra **2,27** nos atrasados (escala de 1 a 5). A mediana cai de 5 para 1. A proporção de notas ruins (1 ou 2) sobe de 8,7% para **62,4%** (7,1 vezes), e a de notas máximas cai de 62,7% para 16,5%. Os pedidos atrasados são 6,7% dos pedidos avaliados, mas geram **34,0% de todas as notas ruins**.

**Discussão (Engenharia de Produção)**

A Fig. 9 mostra uma relação de "dose e resposta": a nota já cai no primeiro dia de atraso (3,30 entre 1 e 3 dias depois), despenca a partir do 4º dia (2,10) e se estabiliza perto de 1,7 a partir de 8 dias. Do lado da antecipação, entregar muito antes não melhora a nota: 21 dias ou mais antes dá 4,33, praticamente o mesmo que 11 a 20 dias antes (4,36); entregas no limite do prazo (0 a 3 dias antes) já têm nota um pouco menor (4,12).

A satisfação está ligada a **cumprir a promessa**, e não a superá-la com muita folga. Isso reforça a P1 e a P2: alongar o prazo prometido nas rotas e nos meses críticos custa pouco em satisfação (quem recebe antes avalia bem) e evita a grande perda de nota que o atraso causa. A folga, porém, não deve crescer sem critério: um prazo longo demais pode afastar o cliente no momento da compra, efeito que esta base não mede, porque só registra quem comprou.

**Cuidados na interpretação:** a nota avalia a experiência inteira (produto, atendimento e entrega), e a comparação é descritiva; os pedidos atrasados podem diferir dos demais em outros pontos, como região e tipo de produto. Mesmo assim, a diferença de 2 pontos e o padrão por faixa de atraso são grandes e consistentes.

#### 7.2.6 Verificação da hipótese

Os cinco critérios definidos na seção 1.2, antes da análise, são confrontados com os resultados. A tabela é montada com as variáveis calculadas acima.

In [ ]:
# C1 - UF com a maior taxa de atraso x taxa nacional
uf_mais_critica = atraso_por_uf.index[0]
razao_uf = atraso_por_uf['vezes_a_taxa_nacional'].iloc[0]

# C2 - mês com a maior taxa x mediana mensal
mes_mais_critico = atraso_por_mes['taxa_atraso'].idxmax()
razao_mes = atraso_por_mes['vezes_a_mediana'].max()

# C3 - participação do transporte e efeito da postagem atrasada
participacao_transporte = participacao_etapas['dias_transporte']

if razao_uf >= 2:
    c1 = 'Sim'
else:
    c1 = 'Não'

if razao_mes >= 2:
    c2 = 'Sim'
else:
    c2 = 'Não'

if participacao_transporte > 50 and razao_postagem >= 2:
    c3 = 'Sim'
else:
    c3 = 'Não'

if razao_interestadual >= 1.5:
    c4 = 'Sim'
else:
    c4 = 'Não'

if diferenca_nota >= 1:
    c5 = 'Sim'
else:
    c5 = 'Não'

verificacao_hipotese = pd.DataFrame({
    'critério': ['C1 (P1)', 'C2 (P2)', 'C3 (P3)', 'C4 (P4)', 'C5 (P5)'],
    'regra definida na seção 1.2': [
        'UF mais crítica com taxa >= 2 vezes a nacional',
        'Algum mês com taxa >= 2 vezes a mediana mensal',
        'Transporte > 50% do tempo e postagem atrasada >= 2 vezes a taxa',
        'Interestadual >= 1,5 vez a taxa do envio local',
        'Nota média dos atrasados pelo menos 1 ponto menor'],
    'resultado obtido': [
        f'{uf_mais_critica}: {razao_uf:.1f} vezes a taxa nacional',
        f'{mes_mais_critico}: {razao_mes:.1f} vezes a mediana',
        f'Transporte = {participacao_transporte:.0f}% do tempo; postagem atrasada = {razao_postagem:.1f} vezes a taxa',
        f'Interestadual = {razao_interestadual:.2f} vezes a taxa local',
        f'Diferença de {diferenca_nota:.2f} ponto'],
    'atendido?': [c1, c2, c3, c4, c5]
})

qtd_atendidos = (verificacao_hipotese['atendido?'] == 'Sim').sum()
if qtd_atendidos == 5:
    conclusao_hipotese = 'CONFIRMADA'
elif qtd_atendidos >= 3:
    conclusao_hipotese = 'PARCIALMENTE CONFIRMADA'
else:
    conclusao_hipotese = 'REJEITADA'

print(f'Critérios atendidos: {qtd_atendidos} de 5  ->  hipótese {conclusao_hipotese}')
verificacao_hipotese

#### 7.2.7 Discussão geral

**A hipótese H foi confirmada (5 de 5 critérios).** Os atrasos de entrega da Olist não são aleatórios:

- têm **endereço**: o Nordeste concentra o risco (AL, MA, SE, PI e CE com mais do que o dobro da taxa nacional), enquanto SP e RJ concentram o volume (metade de todos os pedidos atrasados) — P1;
- têm **época**: a Black Friday de nov/2017 (pico de demanda) e o choque de fev–mar/2018 (pico sem aumento de volume) — P2;
- têm **etapa**: o transporte explica 74% do tempo de entrega e 87% do tempo a mais dos pedidos atrasados, e a postagem atrasada do vendedor é um fator de risco controlável — P3;
- têm **rota**: o envio interestadual atrasa 1,79 vez mais que o local, custa 75% mais de frete e leva quase o dobro do tempo — P4;
- e **custam caro**: o atraso tira 2 pontos da nota média e responde por um terço das notas ruins — P5.

**O fio condutor das cinco respostas é a calibração do prazo prometido.** Em todas as perguntas, o atraso dependeu não só de quanto a entrega demora, mas de como a promessa foi definida em relação à distribuição real dos tempos: as UFs em que a promessa não cobre o P90 atrasam mais que o dobro (P1); nos meses em que o tempo real piorou, a promessa não foi alongada (P2); as rotas médias receberam menos folga que as longas (P4). E a P5 mostra que entregar antes do prometido não reduz a satisfação. Isso torna a recalibração da promessa por rota e por mês a alavanca de **menor custo** do diagnóstico, porque é uma mudança de parâmetro, sem investimento físico.

| # | Ação sugerida | Tipo de ação | Base |
|---|---|---|---|
| 1 | Definir o prazo prometido pelo P90 do tempo real de cada UF ou rota, revisado mês a mês | Parâmetro do sistema (custo baixo) | P1, P2, P4, P5 |
| 2 | Acordo de nível de serviço (SLA) com transportadoras, com o tempo de transporte como indicador por rota (prioridade: rotas para o RJ e para o Nordeste) | Contrato | P1, P3, P4 |
| 3 | Monitorar a postagem atrasada do vendedor como alerta antecipado de atraso | Processo / indicador | P3 |
| 4 | Plano de capacidade e prazos especiais para picos previsíveis (Black Friday) e alerta quando a taxa mensal passar de 2 vezes a mediana | Planejamento | P2 |
| 5 | Atrair vendedores e estoques fora de SP, perto dos clientes de maior risco | Estrutura da rede (custo alto, longo prazo) | P4 |

**As três dimensões do MVP foram atendidas:** a **viabilidade técnica** foi demonstrada pelo pipeline executado de ponta a ponta, com os dados gravados em camadas e relidos do disco; a **viabilidade financeira**, pelo custo zero (só software gratuito) e pelo pouco espaço ocupado (cerca de 123 MB, seção 8.4); e a **desejabilidade**, pelas cinco respostas ligadas a decisões concretas do gestor de logística. O principal limite do diagnóstico é que as comparações são descritivas (mostram associação, não provam causa) e que a base não identifica a transportadora, o que impede apontar quem é responsável pelo gargalo do transporte.

### 7.3 Linha de tomada de decisão (o MVP como sistema de suporte à decisão)

Um sistema de suporte à decisão (SSD) organiza dados, modelos e regras para ajudar o gestor a escolher entre alternativas, sem substituí-lo. O MVP vira um SSD quando os indicadores das perguntas P1 a P5 alimentam **regras de decisão explícitas**. A linha proposta segue as fases do processo decisório de Simon — inteligência, concepção e escolha — e fecha o ciclo com o monitoramento:

```
 INTELIGÊNCIA              CONCEPÇÃO (gatilhos)              ESCOLHA (decisões)                    MONITORAMENTO
 pipeline mensal   --->    G1 promessa < P90 da UF    --->   D1 recalibrar a promessa       --->   taxa de atraso,
 (camada ouro +            G2 taxa da UF >= 2x nacional      D2 SLA com transportadoras            nota média e
  indicadores P1 a P5)     G3 UF no Pareto dos atrasos       D3 plano de pico                      prazo prometido
                           G4 taxa do mês >= 2x mediana      D4 alerta ao cliente e ao vendedor    no mês seguinte
                           G5 postagem fora do prazo
```

| Gatilho (indicador) | Limite | Decisão sugerida | Quem decide | Frequência |
|---|---|---|---|---|
| **G1** Prazo médio prometido da UF × P90 do tempo real de entrega | promessa abaixo do P90 | **D1** Recalibrar a promessa da UF para o P90 | Plataforma (regra automática) | Mensal |
| **G2** Taxa de atraso da UF | pelo menos 2 vezes a nacional | Prioridade alta: D1 e auditoria da rota | Gestor de logística | Mensal |
| **G3** UF entre as que somam 80% dos atrasos, com taxa acima da nacional | Pareto 80% | **D2** Negociar SLA ou redistribuir volume entre transportadoras | Gestor de logística | Trimestral |
| **G4** Taxa de atraso do mês | pelo menos 2 vezes a mediana mensal | **D3** Plano de pico: capacidade extra e prazo temporário maior | Gestor de operações | Mensal e antes da Black Friday |
| **G5** Postagem do vendedor | depois do prazo limite | **D4** Avisar o cliente e cobrar o vendedor | Atendimento e gestão de vendedores | Diária |

Os limites são os mesmos critérios da hipótese (seção 1.2). A seguir, as regras G1 a G3 são aplicadas aos dados para gerar o **painel de decisão por UF**, e a decisão D1 é testada em um **simulador de cenários**. Por fim, a seção 7.3.3 desce ao nível do **pedido**: modelos de machine learning estimam o risco de atraso de cada pedido, para decidir quais acompanhar primeiro.

In [ ]:
# ------------------------------------------------------------
# 7.3.1 Painel de decisão por UF de destino (gatilhos G1, G2 e G3)
# ------------------------------------------------------------
painel_uf = atraso_por_uf[['pedidos', 'taxa_atraso', 'pct_dos_atrasos',
                           'dias_prometidos_medio', 'p90_dias_entrega']].copy()

# G1: o prazo médio prometido não cobre o P90 do tempo real de entrega
painel_uf['G1_promessa_insuficiente'] = painel_uf['dias_prometidos_medio'] < painel_uf['p90_dias_entrega']

# G2: risco alto = taxa de atraso de pelo menos 2 vezes a nacional
painel_uf['G2_risco_alto'] = painel_uf['taxa_atraso'] >= 2 * taxa_nacional

# G3: volume alto = UF entre as que somam 80% dos atrasos (Pareto, P1) e com taxa acima da nacional
ufs_pareto_80 = list(pareto_uf.index[:qtd_ufs_80])
painel_uf['G3_volume_alto'] = painel_uf.index.isin(ufs_pareto_80) & (painel_uf['taxa_atraso'] > taxa_nacional)

# Decisão sugerida (as regras mais específicas vêm por último e prevalecem)
painel_uf['decisao_sugerida'] = 'Monitorar'
painel_uf.loc[painel_uf['G1_promessa_insuficiente'], 'decisao_sugerida'] = 'D1: recalibrar a promessa para o P90'
painel_uf.loc[painel_uf['G3_volume_alto'], 'decisao_sugerida'] = 'D2: negociar SLA com transportadoras'
painel_uf.loc[painel_uf['G3_volume_alto'] & painel_uf['G1_promessa_insuficiente'], 'decisao_sugerida'] = 'D1 + D2: recalibrar a promessa e negociar SLA'

# Prioridade: alta para risco alto (G2) ou volume alto (G3); média só com promessa insuficiente (G1)
painel_uf['prioridade'] = 'Baixa'
painel_uf.loc[painel_uf['G1_promessa_insuficiente'], 'prioridade'] = 'Média'
painel_uf.loc[painel_uf['G2_risco_alto'] | painel_uf['G3_volume_alto'], 'prioridade'] = 'Alta'

# Ordena: prioridade alta primeiro e, dentro dela, quem gera mais atrasos
ordem_prioridade = {'Alta': 1, 'Média': 2, 'Baixa': 3}
painel_uf['ordem'] = painel_uf['prioridade'].map(ordem_prioridade)
painel_uf = painel_uf.sort_values(['ordem', 'pct_dos_atrasos'], ascending=[True, False])
painel_uf = painel_uf.drop(columns=['ordem'])

painel_uf.to_csv(os.path.join(pasta_evidencias, 'painel_decisao_uf.csv'), encoding='utf-8-sig')

print('UFs por prioridade:')
print(painel_uf['prioridade'].value_counts())
painel_uf[painel_uf['prioridade'] != 'Baixa'].round(1)

In [ ]:
# ------------------------------------------------------------
# 7.3.2 Simulador de cenários (what-if) para a decisão D1
# ------------------------------------------------------------
# Pergunta do gestor: "e se a promessa de cada pedido nunca ficasse abaixo do percentil k
# do tempo real de entrega da sua UF?". Pedidos que já tinham promessa maior não mudam.
# O aumento é arredondado para dias inteiros, porque a data prometida não tem horário (R10).
lista_cenarios = []
lista_cenarios.append({
    'politica': 'Atual',
    'taxa_atraso_pct': taxa_nacional,
    'prazo_medio_prometido': prazo_medio_prometido,
    'aumento_medio_dias': 0.0,
    'pct_pedidos_com_promessa_maior': 0.0,
    'aumento_nos_pedidos_afetados': 0.0,
    'atrasos_evitados': 0
})

for percentil in [75, 80, 85, 90, 95]:
    # tempo-alvo de cada UF = percentil k do tempo real de entrega
    tempo_alvo_por_uf = base_analise.groupby('uf_cliente')['dias_entrega_total'].quantile(percentil / 100)
    tempo_alvo = base_analise['uf_cliente'].map(tempo_alvo_por_uf)

    # dias a mais na promessa (a regra só aumenta, nunca reduz)
    dias_a_mais = np.ceil(tempo_alvo - base_analise['dias_prometidos'])
    dias_a_mais = dias_a_mais.clip(lower=0)

    # com a promessa nova, o pedido ainda chegaria depois da data prometida?
    atraso_simulado = np.where(base_analise['dias_atraso'] - dias_a_mais > 0, 1, 0)

    lista_cenarios.append({
        'politica': 'Promessa mínima = P' + str(percentil),
        'taxa_atraso_pct': 100 * atraso_simulado.mean(),
        'prazo_medio_prometido': (base_analise['dias_prometidos'] + dias_a_mais).mean(),
        'aumento_medio_dias': dias_a_mais.mean(),
        'pct_pedidos_com_promessa_maior': 100 * (dias_a_mais > 0).mean(),
        'aumento_nos_pedidos_afetados': dias_a_mais[dias_a_mais > 0].mean(),
        'atrasos_evitados': base_analise['flag_atraso'].sum() - atraso_simulado.sum()
    })

cenarios_promessa = pd.DataFrame(lista_cenarios)
cenarios_promessa.to_csv(os.path.join(pasta_evidencias, 'cenarios_promessa.csv'), index=False, encoding='utf-8-sig')
cenarios_promessa.round(2)

In [ ]:
# Fig. 10 - o trade-off da decisão D1: menos atrasos custam um prazo prometido maior
rotulos_cenarios = ['Atual', 'P75', 'P80', 'P85', 'P90', 'P95']
cores_prazo = [cor_cinza, cor_azul, cor_azul, cor_azul, cor_azul, cor_azul]   # cinza = situação atual

plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
cores_atraso = [cor_cinza, cor_laranja, cor_laranja, cor_laranja, cor_laranja, cor_laranja]
barras = plt.bar(rotulos_cenarios, cenarios_promessa['taxa_atraso_pct'], color=cores_atraso)
plt.bar_label(barras, fmt='%.2f%%', padding=2)
plt.margins(y=0.12)
plt.ylabel('% dos pedidos')
plt.title('Taxa de atraso simulada')

plt.subplot(1, 2, 2)
barras = plt.bar(rotulos_cenarios, cenarios_promessa['prazo_medio_prometido'], color=cores_prazo)
plt.bar_label(barras, fmt='%.1f', padding=2)
plt.margins(y=0.12)
plt.ylabel('Dias')
plt.title('Prazo médio prometido')

plt.suptitle('Fig. 10 – Simulador da decisão D1: promessa mínima no percentil k do tempo real da UF', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(pasta_evidencias, 'fig10_simulador_promessa.png'), dpi=150, bbox_inches='tight')
plt.show()

**Leitura para a decisão**

- **Painel por UF:** as regras G1 a G3 classificam 9 UFs como prioridade alta — RJ, BA, SC e ES pelo volume de atrasos (G3) e AL, MA, SE, PI e CE pelo risco (G2) —, 4 como prioridade média (PA, PB, RN e RR, só com a promessa abaixo do P90) e 14 para monitoramento. A decisão mais importante é a do RJ: recalibrar a promessa (26,3 dias de promessa média contra 29,2 dias de P90) e negociar SLA, porque a UF gera sozinha 22,8% dos atrasos.
- **Simulador (decisão D1):** quanto maior o percentil usado como promessa mínima, menor a taxa de atraso e maior o prazo prometido. Com o P90, a taxa nacional cairia de 6,85% para 4,86% (1.895 atrasos evitados, 29% a menos), e o prazo médio prometido passaria de 23,7 para 26,0 dias; 41,6% dos pedidos teriam a promessa aumentada, em média em 5,6 dias. Com o P80, o ganho é menor (6,15%), mas o prazo quase não muda (24,3 dias).
- **Quem escolhe o ponto da curva é o gestor:** o SSD mostra o custo de cada opção, mas a escolha envolve um julgamento de valor. Prazos maiores reduzem o atraso, mas podem afastar o cliente no momento da compra, efeito que esta base não mede. Uma escolha razoável é começar pelo P85 ou pelo P90 nas 9 UFs de prioridade alta e acompanhar, no mês seguinte, a taxa de atraso, a nota média e a conversão de vendas.

**Cuidados:** o percentil foi calculado com todo o período (2016–2018) e aplicado aos mesmos pedidos, ou seja, o resultado é "dentro da amostra". Em operação, ele deve ser calculado com os últimos meses e aplicado aos pedidos seguintes. A simulação também supõe que o tempo real de entrega não muda quando a promessa muda.

### 7.3.3 Machine learning: risco de atraso de cada pedido

As regras G1 a G5 e o simulador trabalham no nível da **UF** e do **mês**: dizem *onde* e *quando* agir, mas tratam igual todos os pedidos de uma mesma UF. Esta seção usa machine learning para descer ao nível do **pedido**: estimar, para cada pedido novo, o **risco de atraso** (uma probabilidade de 0% a 100%). Com esse risco, a equipe decide **quais pedidos acompanhar primeiro**, antes de o atraso acontecer.

| | Sem machine learning (diagnóstico) | Com o risco de cada pedido |
|---|---|---|
| **Nível da decisão** | UF e mês | Pedido |
| **O que a equipe recebe** | Lista de UFs prioritárias | Lista diária dos pedidos com maior risco |
| **Quando age** | Depois do atraso (reclamação, nota baixa) | Antes: cobra o vendedor, aciona a transportadora, avisa o cliente |

**Como o teste foi montado**

- **Pergunta para o modelo:** "este pedido vai atrasar?" (coluna `flag_atraso`). O modelo devolve uma probabilidade, que é o risco de atraso.
- **Dois momentos de decisão:** **na compra**, quando ainda há mais tempo para agir; e **na postagem**, quando o vendedor entrega o pacote à transportadora e já se sabe quanto tempo ele levou.
- **Divisão por tempo:** treino = pedidos comprados em 2017; teste = pedidos de jan–ago/2018 (mesmo período da P2). Sortear os pedidos ao acaso usaria o futuro para prever o passado.
- **Variáveis:** só o que se sabe no momento da decisão (tabela abaixo). O histórico da rota e do vendedor usa apenas entregas **concluídas antes da data da compra**, com um dia de margem, porque a camada ouro guarda a data da compra sem o horário. O tempo de transporte do próprio pedido e a nota ficam de fora: só existem depois da entrega.
- **Três modelos básicos** (scikit-learn), comparados com uma **regra simples, sem machine learning**, tirada do próprio diagnóstico:
  - **Regra simples:** na compra, ordena os pedidos pela taxa de atraso da UF de destino em 2017 (P1); na postagem, coloca primeiro os pedidos postados fora do prazo (P3, gatilho G5) e, depois, segue a mesma ordem por UF.
  - **Regressão logística:** cada variável soma ou subtrai "pontos de risco", e o total vira uma probabilidade.
  - **Árvore de decisão:** perguntas de sim ou não em sequência (no máximo 5 níveis).
  - **Random forest:** 300 árvores diferentes; o risco é a média delas.
- **Métricas (calculadas no teste):**
  - **Captura nos 10% de maior risco:** que parte de todos os atrasos está entre os 10% de pedidos com maior risco. É a medida ligada à decisão: se a equipe só consegue acompanhar 1 em cada 10 pedidos, quantos atrasos ela encontra? Escolhendo ao acaso, encontraria 10%.
  - **Taxa de atraso nesses 10%:** quantos dos pedidos sinalizados de fato atrasaram (a média do teste é de 7,8%).
  - **AUC:** a chance de o modelo dar um risco maior a um pedido que atrasou do que a um que chegou no prazo (0,5 = sorteio; 1 = perfeito).

| Variável | O que é | Na compra | Na postagem |
|---|---|:-:|:-:|
| `dias_prometidos` | Prazo prometido ao cliente, em dias | ✓ | ✓ |
| `valor_produtos`, `valor_frete`, `qtd_itens` | Valor, frete e quantidade de itens do pedido | ✓ | ✓ |
| `uf_cliente`, `uf_vendedor`, `escopo_envio` | Destino, origem e escopo do envio | ✓ | ✓ |
| `rota_dias_entrega` | Tempo médio de entrega da rota (UF do vendedor → UF do cliente) no histórico | ✓ | ✓ |
| `folga_prometida` | Prazo prometido menos o tempo médio da rota | ✓ | ✓ |
| `vendedor_dias_postagem` | Tempo médio que o vendedor levou para postar, no histórico | ✓ | ✓ |
| `dias_ate_postar` | Dias entre a compra e a postagem deste pedido | | ✓ |
| `flag_postagem_atrasada` | O vendedor postou depois do prazo limite (R11)? | | ✓ |
| `folga_na_postagem` | Prazo prometido menos os dias até postar e menos o tempo médio de transporte da rota | | ✓ |

In [ ]:
# ------------------------------------------------------------
# 7.3.3 Preparação: datas, histórico da rota e do vendedor e divisão por tempo
# ------------------------------------------------------------
# Bibliotecas de machine learning (scikit-learn), usadas só nesta seção
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

base_ml = base_analise.copy()

# Data da compra (a camada ouro guarda só o dia, sem o horário)
base_ml['data_compra'] = pd.to_datetime(base_ml['id_tempo_compra'].astype(str), format='%Y%m%d')
base_ml['data_compra'] = base_ml['data_compra'].astype('datetime64[ns]')

# Data a partir da qual o resultado da entrega já é conhecido: entrega + 1 dia de margem
# (a margem cobre o horário da compra, que não está na camada ouro)
base_ml['data_resultado_conhecido'] = base_ml['data_compra'] + pd.to_timedelta(base_ml['dias_entrega_total'] + 1, unit='D')
base_ml['data_resultado_conhecido'] = base_ml['data_resultado_conhecido'].astype('datetime64[ns]')

# Rota = UF do vendedor -> UF do cliente
base_ml['rota'] = base_ml['uf_vendedor'] + ' > ' + base_ml['uf_cliente']


def media_ate_a_compra(base, chave, coluna):
    """Para cada pedido, a média de `coluna` nos pedidos da mesma `chave` (rota ou vendedor)
    cujo resultado já era conhecido ANTES da data da compra. Usa só o passado."""
    historico = base[[chave, 'data_resultado_conhecido', coluna]].dropna()
    historico = historico.sort_values('data_resultado_conhecido')
    historico['qtd'] = historico.groupby(chave).cumcount() + 1       # pedidos acumulados
    historico['soma'] = historico.groupby(chave)[coluna].cumsum()    # soma acumulada

    compras = base[['id_pedido', chave, 'data_compra']].sort_values('data_compra')
    # merge_asof: para cada compra, pega a última linha do histórico com data ANTERIOR à da compra
    juncao = pd.merge_asof(compras, historico[[chave, 'data_resultado_conhecido', 'qtd', 'soma']],
                           left_on='data_compra', right_on='data_resultado_conhecido',
                           by=chave, allow_exact_matches=False)
    juncao = juncao.set_index('id_pedido').reindex(base['id_pedido'])
    return (juncao['soma'] / juncao['qtd']).values


base_ml['rota_dias_entrega'] = media_ate_a_compra(base_ml, 'rota', 'dias_entrega_total')
base_ml['rota_dias_transporte'] = media_ate_a_compra(base_ml, 'rota', 'dias_transporte')
base_ml['vendedor_dias_postagem'] = media_ate_a_compra(base_ml, 'id_vendedor', 'dias_manuseio')

# Variáveis derivadas
base_ml['folga_prometida'] = base_ml['dias_prometidos'] - base_ml['rota_dias_entrega']
base_ml['dias_ate_postar'] = base_ml['dias_aprovacao'] + base_ml['dias_manuseio']
base_ml['folga_na_postagem'] = (base_ml['dias_prometidos'] - base_ml['dias_ate_postar']
                                - base_ml['rota_dias_transporte'])

# Mesmo período da P2 (jan/2017 a ago/2018) e divisão por tempo
base_ml = base_ml[(base_ml['ano_mes'] >= '2017-01') & (base_ml['ano_mes'] <= '2018-08')]
treino = base_ml[base_ml['ano_mes'] <= '2017-12'].copy()
teste = base_ml[base_ml['ano_mes'] >= '2018-01'].copy()

dias_no_teste = (teste['data_compra'].max() - teste['data_compra'].min()).days + 1

print('Pedidos de treino (jan-dez/2017):', len(treino), f"| taxa de atraso {100 * treino['flag_atraso'].mean():.2f}%")
print('Pedidos de teste (jan-ago/2018) :', len(teste), f"| taxa de atraso {100 * teste['flag_atraso'].mean():.2f}%")
print(f'Pedidos por dia no teste        : {len(teste) / dias_no_teste:.0f}')
print()
print('Pedidos do teste sem histórico (vendedor ou rota novos):')
print(teste[['vendedor_dias_postagem', 'rota_dias_entrega']].isnull().sum())

In [ ]:
# ------------------------------------------------------------
# 7.3.4 Três modelos de risco de atraso, nos dois momentos de decisão
# ------------------------------------------------------------
variaveis_compra = ['dias_prometidos', 'valor_produtos', 'valor_frete', 'qtd_itens',
                    'rota_dias_entrega', 'folga_prometida', 'vendedor_dias_postagem']
variaveis_postagem = variaveis_compra + ['dias_ate_postar', 'flag_postagem_atrasada', 'folga_na_postagem']
variaveis_texto = ['uf_cliente', 'uf_vendedor', 'escopo_envio']

y_treino = treino['flag_atraso']
y_teste = teste['flag_atraso']


def captura(y, risco, fracao):
    """Parte dos atrasos que está entre os `fracao` (ex.: 0.10 = 10%) pedidos de maior risco,
    e a taxa de atraso dentro desse grupo."""
    quantidade = int(round(len(risco) * fracao))
    maiores = np.argsort(-np.asarray(risco), kind='stable')[:quantidade]
    y = np.asarray(y)
    return y[maiores].sum() / y.sum(), y[maiores].mean()


# Regra simples (sem ML): taxa de atraso da UF de destino em 2017 (P1)
taxa_uf_treino = treino.groupby('uf_cliente')['flag_atraso'].mean()
regra_compra = teste['uf_cliente'].map(taxa_uf_treino).fillna(y_treino.mean())
# Na postagem: primeiro os postados fora do prazo (G5); depois, a mesma ordem por UF
regra_postagem = 10 * teste['flag_postagem_atrasada'].fillna(0) + regra_compra

lista_metricas = []
riscos = {}          # risco previsto por momento e modelo (usado nas figuras)
modelos_finais = {}  # random forest de cada momento (usada nas faixas de risco)

for momento, variaveis, regra in [('Na compra', variaveis_compra, regra_compra),
                                  ('Na postagem', variaveis_postagem, regra_postagem)]:
    # Os modelos só entendem números: cada categoria de texto vira uma coluna de 0 e 1
    x_treino = pd.get_dummies(treino[variaveis + variaveis_texto], columns=variaveis_texto, dtype=int)
    x_teste = pd.get_dummies(teste[variaveis + variaveis_texto], columns=variaveis_texto, dtype=int)
    x_teste = x_teste.reindex(columns=x_treino.columns, fill_value=0)   # mesmas colunas do treino

    # Pedido sem histórico (vendedor ou rota novos) ou com etapa anulada (R9): mediana do treino
    medianas_treino = x_treino.median()
    x_treino = x_treino.fillna(medianas_treino)
    x_teste = x_teste.fillna(medianas_treino)

    modelos = {
        'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        'Árvore de decisão': DecisionTreeClassifier(max_depth=5, min_samples_leaf=100, random_state=42),
        'Random forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=50,
                                                max_features='sqrt', n_jobs=-1, random_state=42)
    }

    riscos[(momento, 'Regra simples (sem ML)')] = np.asarray(regra)
    for nome_modelo, modelo in modelos.items():
        modelo.fit(x_treino, y_treino)
        riscos[(momento, nome_modelo)] = modelo.predict_proba(x_teste)[:, 1]   # risco de 0 a 1

    modelos_finais[momento] = (modelos['Random forest'], x_teste)

    for nome_modelo in ['Regra simples (sem ML)', 'Regressão logística', 'Árvore de decisão', 'Random forest']:
        risco = riscos[(momento, nome_modelo)]
        captura_10, taxa_10 = captura(y_teste, risco, 0.10)
        lista_metricas.append({
            'momento': momento,
            'modelo': nome_modelo,
            'captura_nos_10pct (%)': 100 * captura_10,
            'taxa_de_atraso_nos_10pct (%)': 100 * taxa_10,
            'AUC': roc_auc_score(y_teste, risco)
        })

metricas_risco = pd.DataFrame(lista_metricas)
metricas_risco.to_csv(os.path.join(pasta_evidencias, 'metricas_modelos_risco.csv'), index=False, encoding='utf-8-sig')
metricas_risco.round(3)

In [ ]:
# ------------------------------------------------------------
# O que mais pesa no risco (random forest na postagem): importância por permutação
# ------------------------------------------------------------
# Embaralha uma variável de cada vez no teste e mede quanto a AUC cai.
# Quanto maior a queda, mais o modelo depende daquela variável.
floresta_postagem, x_teste_postagem = modelos_finais['Na postagem']
auc_original = roc_auc_score(y_teste, floresta_postagem.predict_proba(x_teste_postagem)[:, 1])
gerador = np.random.RandomState(42)

nomes_variaveis = {
    'dias_prometidos': 'Prazo prometido', 'valor_produtos': 'Valor dos produtos',
    'valor_frete': 'Valor do frete', 'qtd_itens': 'Quantidade de itens',
    'rota_dias_entrega': 'Tempo típico da rota', 'folga_prometida': 'Folga da promessa sobre a rota',
    'vendedor_dias_postagem': 'Tempo típico do vendedor para postar', 'dias_ate_postar': 'Dias até postar',
    'flag_postagem_atrasada': 'Postou fora do prazo', 'folga_na_postagem': 'Prazo que sobra na postagem',
    'uf_cliente': 'UF do cliente', 'uf_vendedor': 'UF do vendedor', 'escopo_envio': 'Escopo do envio'
}

lista_importancia = []
for variavel in variaveis_postagem + variaveis_texto:
    # as colunas de 0 e 1 de uma mesma categoria (ex.: todas as UFs) são embaralhadas juntas
    colunas = [c for c in x_teste_postagem.columns if c == variavel or c.startswith(variavel + '_')]
    quedas = []
    for repeticao in range(3):
        x_embaralhado = x_teste_postagem.copy()
        ordem = gerador.permutation(len(x_embaralhado))
        x_embaralhado[colunas] = x_teste_postagem[colunas].values[ordem]
        auc_embaralhada = roc_auc_score(y_teste, floresta_postagem.predict_proba(x_embaralhado)[:, 1])
        quedas.append(auc_original - auc_embaralhada)
    lista_importancia.append({'variavel': nomes_variaveis[variavel], 'queda_da_AUC': np.mean(quedas)})

importancia_risco = pd.DataFrame(lista_importancia).sort_values('queda_da_AUC', ascending=False)
importancia_risco.round(4)

In [ ]:
# Fig. 11 - quantos atrasos cada modelo encontra olhando 10% dos pedidos, e o que mais pesa no risco
nomes_curtos = ['Regra simples (sem ML)', 'Regressão logística', 'Árvore de decisão', 'Random forest']
tabela_captura = metricas_risco.pivot(index='modelo', columns='momento', values='captura_nos_10pct (%)')
tabela_captura = tabela_captura.loc[nomes_curtos[::-1]]     # random forest em cima

plt.figure(figsize=(13, 4.8))

plt.subplot(1, 2, 1)
posicoes = np.arange(len(tabela_captura))
barras_compra = plt.barh(posicoes + 0.2, tabela_captura['Na compra'], height=0.38, color='#9cc3f0', label='Na compra')
barras_postagem = plt.barh(posicoes - 0.2, tabela_captura['Na postagem'], height=0.38, color=cor_azul, label='Na postagem')
plt.bar_label(barras_compra, fmt='%.1f%%', padding=3, fontsize=9)
plt.bar_label(barras_postagem, fmt='%.1f%%', padding=3, fontsize=9)
plt.axvline(10, color=cor_texto, linestyle='--', linewidth=1.2, label='Ao acaso (10%)')
plt.yticks(posicoes, tabela_captura.index)
plt.xlim(0, 56)
plt.grid(axis='x', alpha=0.4)
plt.grid(axis='y', visible=False)
plt.xlabel('% de todos os atrasos encontrados')
plt.title('Atrasos encontrados olhando os 10% de maior risco')
plt.legend(loc='upper right', fontsize=9)

plt.subplot(1, 2, 2)
maiores_pesos = importancia_risco.head(8).iloc[::-1]
plt.barh(maiores_pesos['variavel'], maiores_pesos['queda_da_AUC'], color=cor_azul)
plt.grid(axis='x', alpha=0.4)
plt.grid(axis='y', visible=False)
plt.xlabel('Queda da AUC quando a variável é embaralhada')
plt.title('O que mais pesa no risco (random forest, na postagem)')

plt.suptitle('Fig. 11 – Risco de atraso por pedido: três modelos x regra simples (teste: jan–ago/2018)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(pasta_evidencias, 'fig11_modelos_risco_atraso.png'), dpi=150, bbox_inches='tight')
plt.show()

**Leitura da comparação dos modelos**

- **O machine learning encontra mais atrasos que a regra simples, com o mesmo esforço.** Olhando os 10% de pedidos de maior risco, a *random forest* encontra 26,5% de todos os atrasos na compra e 35,8% na postagem. A regra simples do diagnóstico encontra 17,9% e 27,7%, e a escolha ao acaso, 10%. No teste, são cerca de 350 atrasos a mais encontrados na compra e 330 na postagem, acompanhando o mesmo número de pedidos.
- **Random forest e regressão logística praticamente empatam.** A *random forest* teve a maior AUC nos dois momentos (0,70 na compra e 0,76 na postagem) e a maior captura na postagem; a regressão logística, mais fácil de explicar, ficou a menos de 1 ponto percentual dela nos dois momentos (na compra, 0,1 ponto acima). A árvore de decisão sozinha, com no máximo 5 perguntas, ficou atrás das duas.
- **Esperar a postagem melhora a previsão, mas encurta o tempo para agir.** Na postagem, o modelo já sabe quanto tempo o vendedor levou e quanto prazo sobra, e a captura sobe de 26,5% para 35,8%. Os dois momentos se completam: na compra, a ação é sobre o vendedor e a data prometida; na postagem, sobre a transportadora e o cliente.
- **O que mais pesa no risco:** o prazo que sobra na postagem (prazo prometido menos os dias já gastos e o tempo típico de transporte da rota), o tempo típico da rota e a UF do cliente. Valor, frete e quantidade de itens quase não pesam.
- **Limite:** mesmo o melhor modelo deixa de fora cerca de dois terços dos atrasos quando se olha só 10% dos pedidos. Parte do atraso vem de fatores que a base não traz, como a transportadora, e de choques que atingem todos os pedidos ao mesmo tempo (seção 7.3.5).

### 7.3.5 Como o risco previsto apoia a decisão

O modelo não decide nada sozinho: ele **ordena** os pedidos do mais arriscado para o menos arriscado. Quem decide é o gestor, em duas escolhas:

1. **Quantos pedidos acompanhar** (o corte), conforme a capacidade da equipe. A **curva de captura** mostra o que se ganha com cada corte: no eixo horizontal, a parte dos pedidos acompanhados (do maior risco para o menor); no vertical, a parte de todos os atrasos que estão nesse grupo.
2. **O que fazer em cada faixa de risco.** Os pedidos são divididos em três faixas: **alto** (os 10% de maior risco), **médio** (os 20% seguintes) e **baixo** (os 70% restantes). A ação sugerida para cada faixa segue o diagnóstico: o atraso nasce no transporte (P3) e derruba a nota do cliente (P5), por isso vale avisar o cliente antes e acionar quem está com o pacote.

In [ ]:
# ------------------------------------------------------------
# 7.3.5 Faixas de risco e curva de captura (random forest), no teste de 2018
# ------------------------------------------------------------
acoes_sugeridas = {
    ('Na compra', 'Alto'): 'Cobrar do vendedor a postagem no prazo e avaliar uma data prometida mais realista para o pedido',
    ('Na compra', 'Médio'): 'Acompanhar a postagem; alerta automático se passar do prazo limite',
    ('Na compra', 'Baixo'): 'Fluxo normal',
    ('Na postagem', 'Alto'): 'Acompanhar com a transportadora e avisar o cliente antes da data prometida',
    ('Na postagem', 'Médio'): 'Monitorar o rastreio; alerta automático se o pacote parar',
    ('Na postagem', 'Baixo'): 'Fluxo normal',
}

lista_faixas = []
for momento in ['Na compra', 'Na postagem']:
    risco = riscos[(momento, 'Random forest')]
    # posição de cada pedido no ranking de risco (0 = maior risco, 1 = menor)
    posicao = pd.Series(risco).rank(ascending=False, method='first').values / len(risco)
    faixa = np.where(posicao <= 0.10, 'Alto', np.where(posicao <= 0.30, 'Médio', 'Baixo'))
    for nome_faixa in ['Alto', 'Médio', 'Baixo']:
        na_faixa = (faixa == nome_faixa)
        lista_faixas.append({
            'momento': momento,
            'faixa_de_risco': nome_faixa,
            'pedidos': na_faixa.sum(),
            'pct_dos_pedidos': 100 * na_faixa.mean(),
            'pedidos_por_dia': na_faixa.sum() / dias_no_teste,
            'risco_medio_previsto (%)': 100 * risco[na_faixa].mean(),
            'taxa_de_atraso_real (%)': 100 * y_teste.values[na_faixa].mean(),
            'pct_dos_atrasos': 100 * y_teste.values[na_faixa].sum() / y_teste.sum(),
            'acao_sugerida': acoes_sugeridas[(momento, nome_faixa)]
        })

faixas_risco = pd.DataFrame(lista_faixas)
faixas_risco.to_csv(os.path.join(pasta_evidencias, 'faixas_risco_teste.csv'), index=False, encoding='utf-8-sig')

# Curva de captura: de 0% a 100% dos pedidos acompanhados, de 1 em 1 ponto percentual
fracoes = np.arange(0, 101) / 100
curva_captura = pd.DataFrame({'pct_dos_pedidos_acompanhados': 100 * fracoes})
for momento in ['Na compra', 'Na postagem']:
    risco = riscos[(momento, 'Random forest')]
    curva_captura['Random forest ' + momento.lower()] = [100 * captura(y_teste, risco, f)[0] if f > 0 else 0 for f in fracoes]
curva_captura['Regra simples na postagem'] = [100 * captura(y_teste, regra_postagem, f)[0] if f > 0 else 0 for f in fracoes]
curva_captura['Ao acaso'] = 100 * fracoes
curva_captura.to_csv(os.path.join(pasta_evidencias, 'curva_captura_atrasos.csv'), index=False, encoding='utf-8-sig')

print('Pontos da curva de captura (random forest):')
print(curva_captura.set_index('pct_dos_pedidos_acompanhados').loc[[5, 10, 20, 30, 50]].round(1))
faixas_risco.round(1)

In [ ]:
# Fig. 12 - curva de captura (quantos pedidos acompanhar) e taxa de atraso real por faixa de risco
plt.figure(figsize=(13, 4.8))

plt.subplot(1, 2, 1)
eixo_x = curva_captura['pct_dos_pedidos_acompanhados']
plt.plot(eixo_x, curva_captura['Random forest na postagem'], color=cor_laranja, linewidth=2.4, label='Random forest, na postagem')
plt.plot(eixo_x, curva_captura['Random forest na compra'], color=cor_azul, linewidth=2.4, label='Random forest, na compra')
plt.plot(eixo_x, curva_captura['Regra simples na postagem'], color=cor_cinza, linewidth=2, label='Regra simples, na postagem')
plt.plot(eixo_x, curva_captura['Ao acaso'], color=cor_texto, linestyle=':', linewidth=1.5, label='Ao acaso')
for corte, altura_texto in [(10, 94), (20, 86)]:
    valor = curva_captura.loc[corte, 'Random forest na postagem']
    plt.scatter([corte], [valor], color=cor_laranja, zorder=3)
    plt.annotate(f'{corte}% dos pedidos: {valor:.0f}% dos atrasos', xy=(corte, valor), xytext=(2, altura_texto),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color=cor_texto))
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.grid(alpha=0.4)
plt.xlabel('% dos pedidos acompanhados (do maior risco para o menor)')
plt.ylabel('% de todos os atrasos encontrados')
plt.title('Curva de captura: quantos pedidos acompanhar?')
plt.legend(loc='lower right', fontsize=9)

plt.subplot(1, 2, 2)
faixas_postagem = faixas_risco[faixas_risco['momento'] == 'Na postagem']
cores_faixas = [cor_laranja, cor_azul, cor_cinza]
rotulos_faixas = [f"{linha['faixa_de_risco']}\n({linha['pct_dos_pedidos']:.0f}% dos pedidos)" for _, linha in faixas_postagem.iterrows()]
barras = plt.bar(rotulos_faixas, faixas_postagem['taxa_de_atraso_real (%)'], color=cores_faixas)
plt.bar_label(barras, fmt='%.1f%%', padding=2)
plt.axhline(100 * y_teste.mean(), color=cor_texto, linestyle='--', linewidth=1.2,
            label=f'Média do teste ({100 * y_teste.mean():.1f}%)')
plt.margins(y=0.15)
plt.ylabel('% dos pedidos que atrasaram')
plt.title('Taxa de atraso real por faixa de risco (na postagem)')
plt.legend(fontsize=9)

plt.suptitle('Fig. 12 – Do risco previsto à decisão: corte de acompanhamento e faixas de risco (teste: jan–ago/2018)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(pasta_evidencias, 'fig12_curva_e_faixas_de_risco.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# O modelo funciona igual em todos os meses? Captura nos 10% de maior risco DE CADA MÊS
# (random forest, na postagem). Ordenar dentro do mês equivale a ter a mesma capacidade todo mês.
risco_postagem = riscos[('Na postagem', 'Random forest')]
lista_meses = []
for mes in sorted(teste['ano_mes'].unique()):
    do_mes = (teste['ano_mes'] == mes).values
    captura_mes, taxa_mes_10 = captura(y_teste.values[do_mes], risco_postagem[do_mes], 0.10)
    lista_meses.append({
        'mes': mes,
        'pedidos': do_mes.sum(),
        'taxa_de_atraso_do_mes (%)': 100 * y_teste.values[do_mes].mean(),
        'captura_nos_10pct_do_mes (%)': 100 * captura_mes
    })

captura_por_mes = pd.DataFrame(lista_meses)
captura_por_mes.round(1)

**Leitura para a decisão**

- **Quantos pedidos acompanhar:** com a *random forest* na postagem, acompanhar 10% dos pedidos (cerca de 22 por dia no teste) encontra 36% dos atrasos; acompanhar 20% (cerca de 43 por dia) encontra 53%. Cada ponto a mais na curva custa mais pedidos acompanhados que não iam atrasar. O corte depende da capacidade da equipe e do custo de cada ação, e essa escolha é do gestor.
- **O risco previsto tem significado prático:** dos pedidos de risco alto, 28,1% atrasaram, 3,6 vezes a média do teste (7,8%); dos de risco baixo, só 3,9%. O risco médio previsto em cada faixa ficou perto da taxa real (25,0%, 11,5% e 3,8%), então a probabilidade pode ser lida como um número de verdade, e não só como uma ordem.
- **O que cada faixa recebe:** na postagem, os pedidos de risco alto são acompanhados com a transportadora e o cliente é avisado antes da data prometida, o que pode reduzir a perda de nota medida na P5; os de risco médio ficam em monitoramento automático do rastreio; os de risco baixo seguem o fluxo normal. Na compra, a ação é cobrar a postagem do vendedor e avaliar uma data prometida mais realista só para os pedidos de risco alto, em vez de aumentar o prazo de uma UF inteira.
- **Os picos pedem outra ferramenta:** ordenando os pedidos dentro de cada mês, o modelo encontra de 38% a 63% dos atrasos nos meses normais, mas só 33% em fevereiro e 27% em março de 2018 (tabela acima). Nos picos, o atraso veio de um choque que atingiu todos os pedidos, e a resposta adequada é o plano de pico (D3), não a lista de pedidos.
- **Como saber se funciona:** o modelo diz *onde* agir, mas não *se a ação funciona*. O próximo passo é um piloto: metade dos pedidos de risco alto recebe a ação e a outra metade não, e no mês seguinte compara-se o atraso e a nota dos dois grupos (fase de monitoramento do SSD).
- **Cuidados:** é um único corte no tempo (treino em 2017, teste em 2018) e os modelos não passaram por busca de parâmetros; as diferenças pequenas entre a *random forest* e a regressão logística não devem ser lidas como vantagem de uma sobre a outra.

## 8. Autoavaliação e resumo para apresentação

### 8.1 Autoavaliação

#### 8.1.1 Quais perguntas de negócio foram respondidas e quais não foram? Por quê?

As cinco perguntas foram respondidas com dados, tabelas e gráficos, e os cinco critérios da hipótese puderam ser verificados. Três respostas têm limites que precisam ser declarados:

- **P2:** os picos foram identificados e medidos, mas as causas (Black Friday, greve dos Correios) só podem ser levantadas como hipóteses, porque a base não traz a transportadora nem eventos externos.
- **P3:** o gargalo foi localizado na etapa de transporte, mas não foi possível dizer *qual* transportadora é responsável, pela mesma razão.
- **P4:** o escopo do envio foi medido por UF e região, que é uma aproximação da distância. A distância em quilômetros exigiria a tabela de geolocalização (1 milhão de linhas), deixada de fora para manter o MVP simples.

#### 8.1.2 Que limitações dos dados condicionaram os resultados obtidos?

- **Período e abrangência:** os dados vão de 2016 a 2018 e vêm de um único *marketplace*; o cenário logístico atual pode ser diferente.
- **Sem transportadora e sem dados do produto usados:** não há identificação da transportadora, e o peso e o tamanho do produto (que explicam parte do frete) não entraram no MVP.
- **Data estimada sem horário:** obrigou a comparar o atraso por dia (regra R10).
- **Nota de avaliação:** mede a experiência inteira (produto, atendimento e entrega), não só a entrega.
- **Recortes do escopo:** ficaram fora da fato 1.275 pedidos entregues com mais de um vendedor (1,3%), e 632 pedidos da fato não têm avaliação.
- **Sem dados de quem desistiu da compra:** não é possível medir o custo de um prazo prometido longo demais.

#### 8.1.3 Quais decisões técnicas eu tomaria de outra forma se recomeçasse o trabalho?

- Usaria a **plataforma Databricks**, que tem o apoio da disciplina, com tabelas Delta (versionamento e histórico das cargas).
- Gravaria as camadas prata e ouro em **Parquet** em vez de CSV: o Parquet guarda o tipo das colunas (o CEP não perderia o zero à esquerda na releitura) e ocupa menos espaço.
- Definiria os **domínios do catálogo antes da carga** e faria o ETL parar automaticamente quando algum teste de domínio desse "NÃO CONFORME".
- Separaria o trabalho em **notebooks menores** (coleta, ETL, análise), como sugere a estrutura de repositório do edital.

#### 8.1.4 Que extensões seriam necessárias para transformar este MVP em uma solução de uso contínuo?

- **Carga incremental agendada** (por exemplo, diária), com os novos pedidos entrando nas camadas bronze, prata e ouro sem reprocessar tudo.
- **Painel de indicadores** mensais: taxa de atraso por UF e por rota, cobertura do P90 pela promessa e taxa de postagem atrasada por vendedor.
- **Regra de promessa calibrada** (simulada na seção 7.3) aplicada de verdade, com o percentil calculado nos últimos meses de cada rota e testada contra a regra atual, medindo também o efeito sobre as vendas.
- **Risco de atraso em operação (seção 7.3.3):** calcular todo dia o risco dos pedidos novos (na compra) e dos pedidos postados (na postagem), enviar à equipe a lista dos pedidos de risco alto, retreinar o modelo todo mês e medir o efeito das ações com um piloto (metade dos pedidos de risco alto recebe a ação, metade não). Incluir a transportadora e o rastreio do pacote deve aumentar a parte dos atrasos que o modelo encontra, hoje cerca de um terço quando se olha 10% dos pedidos.

#### 8.1.5 Dificuldades encontradas na execução do trabalho

- **Conciliação do grão entre pedidos e itens:** a tabela de itens tem uma linha por item, e a de pedidos, uma linha por pedido. Juntar as duas diretamente repetiria o pedido e inflaria a contagem de atrasos. Foi preciso resumir os itens por pedido (quantidade, preço, frete e prazo limite de postagem) e decidir o que fazer com os 1.275 pedidos entregues com itens de mais de um vendedor: como a data de entrega é única por pedido, o atraso não poderia ser atribuído a um só vendedor, e eles foram excluídos (R7).
- **Datas inconsistentes:** 1.359 pedidos têm a postagem registrada antes da aprovação, e 23 têm a entrega antes da postagem. A decisão foi anular só a etapa afetada (R9), sem perder o pedido inteiro, o que exigiu explicar por que o número de pedidos muda entre a P3 e as demais perguntas.
- **Horário zerado na data estimada:** a data estimada vem sempre com 00:00:00. Comparando data e hora, quem recebeu no próprio dia prometido seria contado como atrasado, e a taxa subiria de 6,85% para 8,20%. A solução foi comparar só o dia (R10).
- **CEP que perde o zero à esquerda:** o problema apareceu duas vezes, na leitura do arquivo bruto e na releitura do CSV da camada ouro, e só foi resolvido de vez ao ler o CEP como texto (`dtype=str`).
- **Avaliações repetidas:** 547 pedidos têm mais de uma avaliação, e 814 `review_id` aparecem em mais de um pedido; foi preciso escolher a avaliação mais recente (R5).
- **Execução reprodutível em outro computador:** o notebook precisou rodar do zero na máquina de outra pessoa, sem conta no Google Drive e sem configuração manual. A solução foi usar caminhos relativos à pasta do projeto, listar as bibliotecas em `requirements.txt` e ter uma segunda fonte para a coleta (a cópia da Olist no GitHub) caso o Kaggle não responda.
- **Escolha do período da P2:** os meses de 2016 e de set–out/2018 têm poucos pedidos e gerariam taxas instáveis; foi preciso justificar o recorte de jan/2017 a ago/2018.

### 8.2 Resumo para apresentação

1. **Problema e pipeline:** onde, quando e por que os pedidos da Olist atrasam, e quanto isso custa em satisfação. Pipeline com custo zero, que roda em qualquer computador: Kaggle → `kagglehub` → pastas em camadas (bronze, prata e ouro) → esquema estrela com 95.195 pedidos entregues, catálogo de dados com 35 atributos e análise de qualidade com 19 testes de domínio, todos conformes.
2. **Onde e quando atrasa:** taxa nacional de 6,85%, mas Alagoas chega a 21,5%, e cinco UFs do Nordeste passam do dobro da média; SP e RJ concentram metade dos atrasos. Picos na Black Friday de nov/2017 (12,5%) e em fev–mar/2018 (até 19,2%).
3. **Por que atrasa:** o transporte é o gargalo (74% do tempo de entrega e 87% do tempo a mais dos atrasados); o envio interestadual atrasa 1,79 vez mais que o local e custa 75% mais de frete; e o prazo prometido não acompanha a distribuição real dos tempos (atraso de 12,0% nas UFs em que a promessa não cobre o P90, contra 5,1% nas demais).
4. **Quanto custa e o que fazer (suporte à decisão):** o atraso derruba a nota de 4,31 para 2,27 e gera um terço das notas ruins, enquanto entregar antes não aumenta a nota. A linha de decisão aponta 9 UFs de prioridade alta e um simulador de cenários: com a promessa mínima no P90 de cada UF, a taxa de atraso cairia de 6,85% para 4,86%, ao custo de 2,3 dias a mais no prazo médio prometido.
5. **Machine learning (decisão por pedido):** três modelos básicos (regressão logística, árvore de decisão e *random forest*), treinados em 2017 e testados em jan–ago/2018, estimam o risco de atraso de cada pedido. Olhando só os 10% de pedidos de maior risco, a *random forest* encontra 26,5% dos atrasos na compra e 35,8% na postagem, contra 17,9% e 27,7% da regra simples do diagnóstico e 10% ao acaso. Dos pedidos de risco alto, 28,1% atrasam (3,6 vezes a média); dos de risco baixo, 3,9%. Com isso, a equipe pode acompanhar cerca de 22 pedidos por dia, acionar a transportadora e avisar o cliente antes, em vez de agir sobre uma UF inteira.

### 8.3 Referências

- OLIST. *Brazilian E-Commerce Public Dataset by Olist*. Kaggle. Disponível em: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce. Licença CC BY-NC-SA 4.0.
- BRASIL. Lei nº 13.709, de 14 de agosto de 2018. Lei Geral de Proteção de Dados Pessoais (LGPD).
- VERDÉLIO, A. Funcionários dos Correios entram em greve nesta segunda (12). *Brasil de Fato*, São Paulo, 12 mar. 2018. Publicado originalmente pela Agência Brasil. Disponível em: https://www.brasildefato.com.br/2018/03/12/funcionarios-dos-correios-entram-em-greve-nesta-segunda-feira-12/.
- GOLDRATT, E. M.; COX, J. *The goal: a process of ongoing improvement*. 3. ed. rev. Great Barrington: North River Press, 2004.
- KIMBALL, R.; ROSS, M. *The Data Warehouse Toolkit: the definitive guide to dimensional modeling*. 3. ed. Indianapolis: Wiley, 2013.
- WANG, R. Y.; STRONG, D. M. Beyond accuracy: what data quality means to data consumers. *Journal of Management Information Systems*, v. 12, n. 4, p. 5-33, 1996.
- Documentação: pandas (pandas.pydata.org), matplotlib (matplotlib.org), scikit-learn (scikit-learn.org) e kagglehub (pypi.org/project/kagglehub).

### 8.4 Registro final do pipeline (evidência de persistência e de custo)

A célula abaixo lista tudo o que ficou gravado nas pastas do projeto, mede o espaço ocupado (viabilidade financeira) e o tempo total de execução (viabilidade técnica). Uma captura de tela desta saída serve como evidência da execução para o repositório.

In [ ]:
espaco_total_mb = 0

print('Arquivos gravados nas pastas do projeto (', pasta_projeto, ')')
for pasta_atual in lista_pastas:
    print()
    print(os.path.relpath(pasta_atual, pasta_projeto) + '/')
    for nome_arquivo in sorted(os.listdir(pasta_atual)):
        tamanho_mb = os.path.getsize(os.path.join(pasta_atual, nome_arquivo)) / (1024 * 1024)
        espaco_total_mb = espaco_total_mb + tamanho_mb
        print(f'   {nome_arquivo:40s} {tamanho_mb:7.2f} MB')

tempo_total_min = (time.time() - inicio_pipeline) / 60

print()
print(f'Espaço total ocupado em disco: {espaco_total_mb:.1f} MB')
print(f'Tempo total de execução do notebook: {tempo_total_min:.1f} minutos (inclui o download da base)')
print('Custo de operação: R$ 0,00 (Python e bibliotecas gratuitas)')